# 6.1 CD8 analysis - hyperexpansion

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from skbio.diversity.alpha import gini_index, shannon, simpson

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
from matplotlib_venn import venn2
from scipy.stats import spearmanr
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from brokenaxes import brokenaxes

#Pseudobulk gene expression
import decoupler as dc
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats

#Statsmodel
from statsmodels.nonparametric.smoothers_lowess import lowess

#Venn
from matplotlib_venn import venn3

#Linear regression
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr

#chi-square
from scipy.stats import chi2_contingency

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

In [ ]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("fontTools").setLevel(logging.WARNING)

# Loading data

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_CD8_annotated.h5mu")

## Color dictionaries

In [ ]:
cluster_color_dict = {
    # --- TN ---
    "0":  "#dbe9f6",
    "1":  "#a1d99b",
    "2":  "#e2d9f3",
    "11": "#637939",

    # --- TCM ---
    "7":  "#fd8d3c",
    "13": "#74c476",

    # --- TCSF ---
    "18": "#1f4fbf",
    "14": "#f768a1",
    "3":  "#3182bd",
    "5":  "#D7BCA6",
    "16": "#737373",

    # --- TEM ---
    "4":  "#6baed6",
    "8":  "#8b0000",

    # --- TE ---
    "12": "#843c39",
    "6":  "#C00000",

    # --- TEMRA ---
    "9":  "#d6616b",
    "10": "#fde0c5",

    # --- TA ---
    "17": "#39FF14",
    "19": "#54278f",
    "20": "#c7c9ff",
    "22": "#c51b8a",

    # --- NK-like ---
    "21": "#F4E3ED",

    # --- GDT cells ---
    "15": "#fdae6b",
}


## Create a new column with cluster_colors by mappin to leiden_1.8
obs_df = sdata["gex"].obs[["leiden_1.8", "L3_ann"]]
obs_df["cluster_color"] = obs_df["leiden_1.8"].astype(str).map(cluster_color_dict)

## Make another dictionary with L3_ann and cluster_color
obs_df["leiden_1.8"] = obs_df["leiden_1.8"].astype(str)
obs_df["cluster_color"] = obs_df["leiden_1.8"].map(cluster_color_dict)

L3_ann_colors = (
    obs_df.groupby("L3_ann")["cluster_color"]
    .agg(lambda x: x.mode().iloc[0])  # most frequent color
    .to_dict()
)

# Work with MS vs NIC
### Exclude the two IC donors

In [ ]:
sdata_temp = sdata.copy()
mu.pp.filter_obs(sdata_temp["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))

In [ ]:
sdata_temp["gex"].obs["catpat"]

# CSF-biased clone count

In [ ]:
## First get a list of CSF-biased clones
### A CSF-biased clones has 0.538 cells in CSF

In [ ]:
obs_df = sdata_temp["gex"].obs.copy()
obs_df = obs_df[~obs_df["clone_handle"].isna()]
obs_df = obs_df[obs_df["total_clone_count"] > 12]
obs_df = obs_df[["catpat", "donor", "clone_handle", "Compartment", "type_x_clone_count"]].drop_duplicates()

#Turn all categories into strings
for col in obs_df.select_dtypes(["category"]):
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

#Save catpat
c_ann = obs_df[["catpat", "donor"]].drop_duplicates().copy()

########## ---- Define CSF-biased clones
## count df
count_df = obs_df.pivot(index=["catpat", "donor", "clone_handle"], columns="Compartment", values="type_x_clone_count").reset_index()
count_df["CSF"] = count_df["CSF"].fillna(0)
count_df["PB"] = count_df["PB"].fillna(0)

## CSF fraction
count_df["CSF_fraction"] = count_df["CSF"] / (count_df["CSF"]+count_df["PB"])

## annotate
count_df["biased_ann"] = np.where(count_df["CSF_fraction"] >= 0.538, "CSF-biased", 
                                  np.where(count_df["CSF_fraction"] == 0, "PB-biased", "Neutral"))

## list
csf_biased_clones = count_df[count_df["biased_ann"] == "CSF-biased"]["clone_handle"].drop_duplicates()
######### ---- 

### Clone counts per ml
obs_df = sdata_temp["gex"].obs.copy()
obs_df = obs_df[~obs_df["clone_handle"].isna()]
obs_df = obs_df[obs_df["total_clone_count"] > 12]
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df = obs_df[obs_df["clone_handle"].isin(csf_biased_clones)] ## Filter on CSF-biased clones
obs_df = obs_df[["catpat", "donor", "clone_handle", "Compartment", "type_x_clone_count"]].drop_duplicates()  #To have a dataframe with unique clones

#Turn all categories into strings
for col in obs_df.select_dtypes(["category"]):
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

## count_df 
count_df = obs_df.groupby("donor")["type_x_clone_count"].sum().reset_index()

## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}

## add csf volume data
count_df["csf_vol"] = count_df["donor"].map(csf_ml)

## normalize
count_df["count_per_ml"] = count_df["type_x_clone_count"] / count_df["csf_vol"]

## annotate
plot_df = count_df.merge(c_ann, how="left")

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

sb.boxplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    hue="catpat",
    log_scale=False,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    hue="catpat",
    log_scale=False,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

## Statistics
MS_vals = plot_df[plot_df["catpat"] == "MS"]["count_per_ml"]
ctrl_vals = plot_df[plot_df["catpat"] == "NIC"]["count_per_ml"]
stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
pval

# Plot: % of cells discovered in CSF over expansion degree

In [ ]:
## Test out making a dataframe first

# Step 1: Data Preparation and Filtering
obs_df = sdata["gex"].obs.copy()

# Make clone_handle categorical
obs_df["clone_handle"] = obs_df["clone_handle"].astype("category")

# Filter for clone_handle in comp_set and remove donors Pt4 and Pt5
# obs_df = obs_df[obs_df["total_clone_count"] > 11]
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[~obs_df["clone_handle"].isna()]
print("I have this many unique clones: ", obs_df["clone_handle"].nunique(), ", across both CSF and PB combined")

# Filter of CSF to make certain that we are working with CSF clones
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
obs_df["clone_handle"] = obs_df["clone_handle"].cat.remove_unused_categories()
obs_df["Compartment"] = obs_df["Compartment"].cat.remove_unused_categories()

#Polish dataframe
obs_df = obs_df[["donor", "catpat", "clone_handle", "total_clone_count", "type_x_clone_count"]].drop_duplicates()

In [ ]:
def expansion_sweep_df_donor2(sdata_temp, catpat="MS", max_expansion=25):

    all_results = []

    for x in range(0, max_expansion + 1):

        # Get relevant columns
        obs_df = sdata_temp["gex"].obs[
             ["catpat", "donor", "Compartment", "clone_handle", "type_x_clone_count", "total_clone_count"] ## By dropping duplicates each clone will only be present once. 
        ].drop_duplicates().copy()

        # Subset on patient group and expansion threshold
        obs_df = obs_df[~obs_df["clone_handle"].isna()]
        obs_df = obs_df[obs_df["catpat"] == catpat]
        obs_df = obs_df[obs_df["total_clone_count"] > x]

        #Turn all categories into strings
        for col in obs_df.select_dtypes(["category"]):
            obs_df[col] = obs_df[col].cat.remove_unused_categories()

        ## Calculate a sum of type_x_clone_count
        obs_df = obs_df.groupby(["catpat", "donor", "Compartment"], observed=True, as_index=False)["type_x_clone_count"].sum()
        # Pivot to determine CSF/PB presence
        obs_df = obs_df.pivot_table(
            index=["donor"],
            columns="Compartment",
            values="type_x_clone_count",
            fill_value=0
        )

        # Calculate %CSF
        obs_df = obs_df.reset_index()
        obs_df["total"] = obs_df["CSF"] + obs_df["PB"]
        obs_df["CSF_pct"] = 100*obs_df["CSF"] / obs_df["total"]

        # Add expansion threshold
        obs_df["expansion_threshold"] = x

        # Add patient category
        obs_df["catpat"] = catpat

        all_results.append(obs_df)

    return pd.concat(all_results, ignore_index=True)

In [ ]:
## Make the dataframes
df_MS = expansion_sweep_df_donor2(sdata_temp, "MS")
df_ctrl = expansion_sweep_df_donor2(sdata_temp, "NIC")

## Concatenate
out_concat = pd.concat([df_MS, df_ctrl])

In [ ]:
plot_df = out_concat[out_concat["expansion_threshold"].isin([0, 5, 10, 15, 20])]

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
sb.boxplot(
    data=plot_df,
    x="expansion_threshold",
    y="CSF_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=plot_df,
    x="expansion_threshold",
    y="CSF_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("% of cells in CSF per donor") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/13_boxplots_catexp_thresholds_percentage_cells.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = [0, 5, 10, 15, 20]

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = plot_df[plot_df["expansion_threshold"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["CSF_pct"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["CSF_pct"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats

## Mean clone-wise percentage of cells in CSF

In [ ]:
def expansion_sweep_clone_wise_mean(sdata_temp, catpat="MS", max_expansion=25):

    all_results = []

    for x in range(0, max_expansion + 1):

        # Get relevant columns
        obs_df = sdata_temp["gex"].obs[
             ["catpat", "donor", "Compartment", "clone_handle", "type_x_clone_count", "total_clone_count"] ## By dropping duplicates each clone will only be present once. 
        ].drop_duplicates().copy()

        # Subset on patient group and expansion threshold
        obs_df = obs_df[~obs_df["clone_handle"].isna()]
        obs_df = obs_df[obs_df["catpat"] == catpat]
        obs_df = obs_df[obs_df["total_clone_count"] > x]

        #Turn all categories into strings
        for col in obs_df.select_dtypes(["category"]):
            obs_df[col] = obs_df[col].cat.remove_unused_categories()

        # Pivot to determine CSF/PB presence
        obs_df = obs_df.pivot_table(
            index=["donor", "clone_handle"],
            columns="Compartment",
            values="type_x_clone_count",
            fill_value=0
        )

        # Calculate %CSF
        obs_df = obs_df.reset_index()
        obs_df["total"] = obs_df["CSF"] + obs_df["PB"]
        obs_df["CSF_pct"] = 100*obs_df["CSF"] / obs_df["total"]

        # Calculate mean clone-wise %CSF per donr
        obs_df = obs_df.groupby("donor")["CSF_pct"].mean().reset_index()
        
        # Add expansion threshold
        obs_df["expansion_threshold"] = x

        # Add patient category
        obs_df["catpat"] = catpat

        all_results.append(obs_df)

    return pd.concat(all_results, ignore_index=True)

In [ ]:
## Make the dataframes
df_MS = expansion_sweep_clone_wise_mean(sdata_temp, "MS")
df_ctrl = expansion_sweep_clone_wise_mean(sdata_temp, "NIC")

## Concatenate
out_concat = pd.concat([df_MS, df_ctrl])

In [ ]:
exp_threshold = [1, 6, 12, 18, 24]
plot_df = out_concat[out_concat["expansion_threshold"].isin(exp_threshold)]

# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
sb.boxplot(
    data=plot_df,
    x="expansion_threshold",
    y="CSF_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=plot_df,
    x="expansion_threshold",
    y="CSF_pct",
    hue="catpat",
    hue_order=["NIC", "MS"],
    dodge=True,
    log_scale=True,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# Remove duplicate legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="Group")

ax.set_xlabel("Expansion thresholds (n cells > x)")
ax.set_ylabel("Mean clone-wise % in CSF") #% of cells in CSF\nnorm. to clone sizes

sb.despine(ax=ax)
plt.tight_layout()

fig.savefig(
    "../../05_plots/13_final_plots/13_boxplots_catexp_thresholds_percentage_cells.pdf",
    bbox_inches='tight',
    format="pdf",
    dpi=400
)

plt.show()

#Save a list of expansion thresholds
threshold_order = exp_threshold

# Initialize list to collect results
stats_list = []

for xtick in threshold_order:
    data_xtick = plot_df[plot_df["expansion_threshold"] == xtick]
    
    MS_vals = data_xtick[data_xtick["catpat"] == "MS"]["CSF_pct"]
    ctrl_vals = data_xtick[data_xtick["catpat"] == "NIC"]["CSF_pct"]
    
    if len(MS_vals) > 0 and len(ctrl_vals) > 0:
        stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
        
        stats_list.append({
            "expansion_threshold": xtick,
            "statistic": stat,
            "p_value": pval,
            "median_MS": MS_vals.median(),
            "median_NIC": ctrl_vals.median(),
            "n_MS": len(MS_vals),
            "n_NIC": len(ctrl_vals)
        })

# Convert to dataframe
df_stats = pd.DataFrame(stats_list)

# Apply Benjamini-Hochberg correction
reject, pvals_bh, _, _ = multipletests(
    df_stats["p_value"],
    alpha=0.05,
    method="fdr_bh"
)

# Add results to dataframe
df_stats["p_value_BH"] = pvals_bh
df_stats["significant_BH"] = reject

df_stats

# Report clonal expansion per cluster
Proportion of each cluster made up of clonally expanded cells at 5 different bins

In [ ]:
#First, generate 5 different bins of biological relevance
bins = [0, 1, 5, 10, 20, float("inf")]
labels = ["1", "2-5", "6-10", "11-20", ">20"]

# Bin the total_clone_count column
sdata["gex"].obs["clone_count_bin"] = pd.cut(
    sdata["gex"].obs["total_clone_count"],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

In [ ]:
sdata["gex"].obs[["clone_handle", "clone_count_bin"]].drop_duplicates()["clone_count_bin"].value_counts()

In [ ]:
# # 🎨 Set global figure parameters
# sc.settings.set_figure_params(dpi=75, facecolor="white")

# # List of donors to highlight individually
# labels = ["1", "2-5","6-10", "11-20", ">20"]

# # 🔲 Set up figure grid (adjust cols as needed)
# n_cols = len(labels)
# n_rows = -(-len(labels) // n_cols)  # Ceiling division

# fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))

# # Flatten axes for easy indexing
# axes = axes.flatten()

# # 🔁 Plot one labels per subplot
# for i, label in enumerate(labels):
#     ax = axes[i]
#     sc.pl.umap(
#         sdata["gex"],
#         color="clone_count_bin",
#         groups=[label],
#         size=10,
#         add_outline=False,
#         legend_fontoutline=2,
#         legend_fontweight='bold',
#         legend_fontsize=6,
#         legend_loc="on data",
#         frameon=False,
#         vmin=0,
#         show=False,
#         ax=ax,
#         title=label,
#     )

# # 🧼 Turn off any unused axes
# for j in range(len(labels), len(axes)):
#     fig.delaxes(axes[j])

# # 💾 Save the combined plot
# plt.tight_layout()
# plt.savefig("../../05_plots/08_AfterAntwerp/68_CD8_ClonalBins.pdf", bbox_inches="tight", dpi=400)
# plt.show()

In [ ]:
# Extract relevant data
df = sdata["gex"].obs[["clone_handle", "clone_count_bin", "leiden_1.8"]].copy()

# Drop missing annotations if any
df = df.dropna(subset=["clone_count_bin", "leiden_1.8"])

# Compute counts
counts = (
    df.groupby(["leiden_1.8", "clone_count_bin"])
    .size()
    .unstack(fill_value=0)
)

# Convert to proportions
proportions = counts.div(counts.sum(axis=1), axis=0)

# custom cmap
colors = ['lightgrey', '#ff7f0e', '#279e68', '#d62728', '#aa40fc']  # matches bin_order
custom_cmap = ListedColormap(colors)

cluster_order = [
    "0", "1", "2", "11",        # TN
    "7", "13",                  # TCM
    "4", "8",                  # TEM
    "12", "6",                 # TE
    "9", "10",                 # TEMRA
    "17", "19", "20", "22",    # TA
    "21",                      # NK-like
    "15",                      # GDT cells

    # moved to end
    "18", "14", "3", "5", "16"
]

#re-index in a specific order
proportions = proportions.reindex(
    [cl for cl in cluster_order if cl in proportions.index]
)

# Plot
proportions.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 3),
    width=0.8,
    colormap=custom_cmap
)

plt.ylabel("Proportion")
plt.xticks(rotation=0)
plt.title("Proportional Contribution of Clone Count Bins per L3 Annotation")
plt.legend(title="Clone Count Bin", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)
sb.despine()
plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/Expansion_over_clusters.pdf")
plt.show()

# Report Gini index per cluster

In [ ]:
###Clean-up.
#Make sure only the indexes recorded in GEX are used in the o*ther modalities too
index_list = sdata_temp["gex"].obs.index.tolist()

## define a cell index
sdata_temp["airr"].obs["cell_index"] = sdata_temp["airr"].obs.index
sdata_temp["prot"].obs["cell_index"] = sdata_temp["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
sdata_temp["airr"].obs["filter_index"] = pd.Categorical(np.where(sdata_temp["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
sdata_temp["prot"].obs["filter_index"] = pd.Categorical(np.where(sdata_temp["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

#Enforce that sdata_temp["airr"].obs and sdata_temp["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(sdata_temp["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(sdata_temp["prot"], 'filter_index', lambda x: x == "GEX_present")

## Make a copy and work in that
sdata_temp_gini = sdata_temp.copy()
sdata_temp_gini["airr"].obs["clone_handle"] = sdata_temp_gini["gex"].obs["clone_handle"].copy()
sdata_temp_gini["airr"].obs["leiden_1.8"] = sdata_temp_gini["gex"].obs["leiden_1.8"].copy()
sdata_temp_gini["airr"].obs["donor"] = sdata_temp_gini["gex"].obs["donor"].copy()
sdata_temp_gini["airr"].obs["catpat"] = sdata_temp_gini["gex"].obs["catpat"].copy()

#Make grouping variables
sdata_temp_gini["airr"].obs["donor_leiden_1.8"] = sdata_temp_gini["airr"].obs["donor"].astype(str) + "-" + sdata_temp_gini["airr"].obs["leiden_1.8"].astype(str)

#filter
mu.pp.filter_obs(
    sdata_temp_gini["airr"],
    "clone_id",
    lambda x: ~pd.isna(x)
)

## %% Calculate the diversity metrics andsave them under a column key. 
ir.tl.alpha_diversity(sdata_temp_gini["airr"], groupby="leiden_1.8", target_col='clone_handle', metric='gini_index', inplace=True, key_added="gini_per_cluster", airr_mod='airr')

# Drop rows with NaN gini index
df_gini = sdata_temp_gini["airr"].obs[["leiden_1.8", "gini_per_cluster"]].drop_duplicates().reset_index()
df_gini = df_gini.dropna(subset=["gini_per_cluster"])
df_gini["gini_per_cluster"] = pd.to_numeric(df_gini["gini_per_cluster"])

# df_gini.sort_values("gini_per_cluster")
# order = df_gini.sort_values("gini_per_cluster",ascending=False)["leiden_1.8"].to_list()
df_gini["leiden_1.8"] = df_gini["leiden_1.8"].astype(str)

# Define the order
cluster_order = [
    "0", "1", "2", "11",        # TN
    "7", "13",                  # TCM
    "4", "8",                  # TEM
    "12", "6",                 # TE
    "9", "10",                 # TEMRA
    "17", "19", "20", "22",    # TA
    "21",                      # NK-like
    "15",                      # GDT cells

    # moved to end
    "18", "14", "3", "5", "16"
]

plt.figure(figsize=(8,5))

sb.barplot(
    data=df_gini,
    x='leiden_1.8',
    y='gini_per_cluster',
    order=cluster_order,
    color='lightgrey'
)

plt.xlabel("leiden_1.8")
plt.ylabel("Gini index")
plt.title("Gini per Leiden cluster")
# edit layout
sb.despine()
ax.grid(False)

plt.tight_layout()
plt.savefig("../../05_plots/11_CD8annotation/17_Gini_per_cluster.pdf", bbox_inches="tight",  dpi=400)
plt.show()

# Percent clonal overlap

In [ ]:
## Reproduce Kavaka et al analysis with clonal overlap among each cluster
#### Of all the CSF clonotypes - which cluster contains the most overlap?

obs_df = sdata["gex"].obs[["donor", "Compartment", "catpat", "leiden_1.8", "clone_handle"]].copy()
obs_df = obs_df[~obs_df["clone_handle"].isna()]

## TCSF cells
obs_csf = obs_df[obs_df["Compartment"]=="CSF"]
list_csf_clones = obs_csf["clone_handle"].drop_duplicates().tolist()

# Ensure CSF clones is a set
csf_clones = set(list_csf_clones)

# Define the clusters to iterate over and the cluster column. 
# These clusters has to be PB clusters
clusters =   [ "0", "1", "2", "11",        # TN
    "7", "13",                  # TCM
    "4", "8",                  # TEM
    "12", "6",                 # TE
    "9", "10",                 # TEMRA
    "17", "19", "20", "22",    # TA
    "21",                      # NK-like
    "15",                      # GDT cells
]
cluster_col = "leiden_1.8"

# Define an empty dataframe
results = []

for cl in clusters:
    # Subset for given cluster in PB
    sub = obs_df[
        (obs_df[cluster_col] == cl) &
        (obs_df["Compartment"] == "PB")
    ]

    cluster_clones = set(sub["clone_handle"].drop_duplicates())

    # Overlap and union
    overlap = csf_clones & cluster_clones
    union = csf_clones | cluster_clones

    pct_csf_in_cluster = (
        len(overlap) / len(csf_clones) * 100
        if len(csf_clones) > 0 else np.nan
    )

    jaccard = (
        len(overlap) / len(union) * 100
        if len(union) > 0 else np.nan
    )

    results.append({
        cluster_col: cl,
        "n_csf_clones": len(csf_clones),
        "n_cluster_clones": len(cluster_clones),
        "n_overlap": len(overlap),
        "pct_csf_in_cluster": pct_csf_in_cluster,
        "jaccard_index": jaccard
    })

df_results = pd.DataFrame(results)



## Make a column plot that describes clonal sharing amongst CD8 T cells
# Ensure correct ordering
# order = ["TN", "TSCM", "TCM", "TEM", "TE", "TAct"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- Plot 1: % CSF in cluster ---
sb.barplot(
    data=df_results,
    x="leiden_1.8",
    y="pct_csf_in_cluster",
     color="#D89A52",
    # order=order,
    ax=axes[0]
)

axes[0].set_title("% CSF clones in cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Percentage")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(False)

# --- Plot 2: Jaccard index ---
sb.barplot(
    data=df_results,
    x="leiden_1.8",
    y="jaccard_index",
    # order=order,
    ax=axes[1]
)

axes[1].set_title("Jaccard index")
axes[1].set_xlabel("PB cluster")
axes[1].set_ylabel("Percentage")
axes[1].tick_params(axis="x", rotation=0)
axes[1].grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/13_final_plots/22_CD8_Jaccard_overlap_leiden_1.8.pdf")
plt.show()

In [ ]:
## Do the opposite - which clusters in CSF share clones with PB? 

obs_df = sdata["gex"].obs[["donor", "Compartment", "catpat", "leiden_1.8", "clone_handle"]].copy()
obs_df = obs_df[~obs_df["clone_handle"].isna()]

## TCSF cells
obs_pb = obs_df[obs_df["Compartment"]=="PB"]
list_pb_clones = obs_pb["clone_handle"].drop_duplicates().tolist()

# Ensure CSF clones is a set
csf_clones = set(list_pb_clones)

# Define the clusters to iterate over and the cluster column. 
# These clusters has to be PB clusters
clusters =   [ "18", "14", "3", "5", "16"]
cluster_col = "leiden_1.8"

# Define an empty dataframe
results = []

for cl in clusters:
    # Subset for given cluster in PB
    sub = obs_df[
        (obs_df[cluster_col] == cl) &
        (obs_df["Compartment"] == "CSF")
    ]

    cluster_clones = set(sub["clone_handle"].drop_duplicates())

    # Overlap and union
    overlap = csf_clones & cluster_clones
    union = csf_clones | cluster_clones

    pct_csf_in_cluster = (
        len(overlap) / len(csf_clones) * 100
        if len(csf_clones) > 0 else np.nan
    )

    jaccard = (
        len(overlap) / len(union) * 100
        if len(union) > 0 else np.nan
    )

    results.append({
        cluster_col: cl,
        "n_csf_clones": len(csf_clones),
        "n_cluster_clones": len(cluster_clones),
        "n_overlap": len(overlap),
        "pct_csf_in_cluster": pct_csf_in_cluster,
        "jaccard_index": jaccard
    })

df_results = pd.DataFrame(results)



## Make a column plot that describes clonal sharing amongst CD8 T cells
# Ensure correct ordering
# order = ["TN", "TSCM", "TCM", "TEM", "TE", "TAct"]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# --- Plot 1: % CSF in cluster ---
sb.barplot(
    data=df_results,
    x="leiden_1.8",
    y="pct_csf_in_cluster",
     color="#D89A52",
    # order=order,
    ax=axes[0]
)

axes[0].set_title("% CSF clones in cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Percentage")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(False)

# --- Plot 2: Jaccard index ---
sb.barplot(
    data=df_results,
    x="leiden_1.8",
    y="jaccard_index",
    # order=order,
    ax=axes[1]
)

axes[1].set_title("Jaccard index")
axes[1].set_xlabel("PB cluster")
axes[1].set_ylabel("Percentage")
axes[1].tick_params(axis="x", rotation=0)
axes[1].grid(False)

plt.tight_layout()
fig.savefig("../../05_plots/13_final_plots/22_CD8_Jaccard_overlap_leiden_1.8_CSF.pdf")
plt.show()

# Barplots - compartment-wise expansion and sharing

#### Step 1: Identify clones with more than X cells total as interesting

In [ ]:
sdata["gex"].obs["select20"] = pd.Categorical(np.where(
    sdata["gex"].obs["total_clone_count"] > 20, "High frequent", "Low frequent"))

sdata["gex"].obs["select10"] = pd.Categorical(np.where(
    sdata["gex"].obs["total_clone_count"] > 10, "High frequent", "Low frequent"))

sdata["gex"].obs["select5"] = pd.Categorical(np.where(
    sdata["gex"].obs["total_clone_count"] > 5, "High frequent", "Low frequent"))

### Step 2: Zoom in on those clones/cells specifically

In [ ]:
#Define a set of clones for fitlering:
set5 = set(sdata["gex"].obs[sdata["gex"].obs["select5"] == "High frequent"]["clone_handle"])
set10 = set(sdata["gex"].obs[sdata["gex"].obs["select10"] == "High frequent"]["clone_handle"])
set20 = set(sdata["gex"].obs[sdata["gex"].obs["select20"] == "High frequent"]["clone_handle"])

#Make a data copy and filter that
sdata_highfrequent = sdata.copy()

#Make clone_handle categorical
sdata_highfrequent["gex"].obs["clone_handle"] = sdata_highfrequent["gex"].obs["clone_handle"].astype("category")

#Filter
# mu.pp.filter_obs(sdata_highfrequent["gex"], "clone_handle", lambda x: (x.isin(set5))) 
mu.pp.filter_obs(sdata_highfrequent["gex"], "clone_handle", lambda x: (x.isin(set20))) 

### Step 3: Calculate compartment-wise frequency

In [ ]:
## Since the total is already set within sdata (total_clone_count) as well as the compartment-wise count (type_x_clone_count), we just enumerate the percentage directly
sdata_highfrequent["gex"].obs["Compartment_pct"] = 100*sdata_highfrequent["gex"].obs["type_x_clone_count"]/sdata_highfrequent["gex"].obs["total_clone_count"] 

##Sanity checks - how many clones do I have?
print("I have this many unique clones: ", len(sdata_highfrequent["gex"].obs["clone_handle"].drop_duplicates()))

#### Step 5: Plot compartment-wise frequency as stacked barplots

In [ ]:
df_input  = sdata_highfrequent["gex"].obs[["clone_handle","Compartment", "Compartment_pct"]]

# Pivot the data so compartments become columns
pivot_table = df_input.pivot_table(
    index="clone_handle",
    columns="Compartment",
    values="Compartment_pct",
    fill_value=0
)

# Optional: Rename columns to include '.pct' for clarity
pivot_table.columns = [f"{col}.pct" for col in pivot_table.columns]

# Sort to show CSF-high clones first
pivot_table = pivot_table.sort_values(by="CSF.pct", ascending=False)

# Get the number of bars (i.e., clone handles) for the title
num_bars = len(pivot_table)

# Plot
fig, ax = plt.subplots(figsize=(15, 3))
x = np.arange(len(pivot_table))
width = 1

# Stacked bars
ax.bar(x, pivot_table["CSF.pct"], width, label="CSF", color="#4B8AC1", edgecolor="none")
ax.bar(x, pivot_table["PB.pct"], width, bottom=pivot_table["CSF.pct"], label="PB", color="#C06A6A", edgecolor="none")

# Formatting
ax.set_title(f"Percentage contribution of different compartments per clone for clones w. more than 20 cells - {num_bars} clones", fontsize=12)
ax.set_xlabel("Clone Handle")
ax.set_ylabel("Percentage")
ax.set_xticks([])  # Remove x-axis labels

# Remove box (spines)
for spine in ["top", "right", "left", "bottom"]:
    ax.spines[spine].set_visible(False)

# Optional: Add y-axis line for clarity
ax.axvline(-0.5, color="black", linewidth=1)

# Legend outside
ax.legend(title="Compartment", bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)

plt.tight_layout()
plt.show()

# Permutation to find a hyperexpansion cutoff 

### Calculate %CSF and %PB from NIC and MS patients separately. 
Simulate drawing, and calculate statistical likelihoods of seeing different clone purities at different clone sizes. At the extreme end, a completely pure clone will - given our background distribution in our data - be either skewed 100% towards CSF or PB

In [ ]:
obs_df = sdata["gex"].obs[["donor", "catpat", "clone_handle", "Compartment"]]

#Filter - exclude Pt4 and Pt5
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]

#calculate the number of cells in CSF for all appropriate donors first (no catpat subset)
compart_sum = obs_df.groupby(["Compartment"]).size().reset_index(name="count")
CSF_cells = compart_sum.loc[compart_sum["Compartment"] == "CSF", "count"].iloc[0]
PB_cells = compart_sum.loc[compart_sum["Compartment"] == "PB", "count"].iloc[0]

#Calculate the fraction of cells in CSF of the data
CSF_pct = CSF_cells / (CSF_cells + PB_cells)
CSF_pct

#For grouped results
compart_sum_grp = obs_df.groupby(["Compartment", "catpat"]).size().reset_index(name="count").pivot(index = "catpat", columns="Compartment", values="count")
compart_sum_grp["total"] = compart_sum_grp["CSF"] + compart_sum_grp["PB"]  
compart_sum_grp["CSF.pct"] = compart_sum_grp["CSF"] / compart_sum_grp["total"] 
compart_sum_grp = compart_sum_grp.reset_index()

In [ ]:
CSF_pct #percentage of the dataset that is CSF cells

In [ ]:
## Given these statistics, make permutations to indicate what is highly significant
# Parameters
p_csf = CSF_pct  # compart_sum.loc[compart_sum["catpat"] == "MS", "CSF.pct"].iloc[0]
alpha = 0.05
max_cells = 80

rows = []

for n in range(1, max_cells + 1):
    for k in range(0, n + 1):
        purity = k / n

        # Binomial test under null
        pval = binomtest(k, n, p_csf, alternative="two-sided").pvalue

        if pval <= alpha:
            rows.append({
                "n_cells": n,
                "purity": purity
            })

sig_df = pd.DataFrame(rows)

In [ ]:
sig_df[sig_df["n_cells"] == 13]

In [ ]:
# Prepare the figure
fig, ax = plt.subplots(figsize=(4, 3.75))

# Plot data points
ax.scatter(
    sig_df["n_cells"],
    sig_df["purity"],
    s=10,
    color="black"
)

ax.axhline(0.538, linestyle="--", linewidth=1)
ax.axvline(13, linestyle="--", linewidth=1)

ax.set_xlabel("Number of cells discovered")
ax.set_ylabel("Fraction of cells in CSF of a given clone")
ax.set_title("Aggregated MS & Ctrl")

fig.tight_layout()

# ---- Save BEFORE show ----
fig.savefig(
    "../../05_plots/13_final_plots/27_clone_size_vs_csf_fraction.pdf",
    bbox_inches="tight",
    format="pdf",
    dpi=400
)

plt.show()

In [ ]:
## Given these statistics, make permutations to indicate what is highly significant
# Parameters
p_csf = compart_sum_grp.loc[compart_sum_grp["catpat"] == "MS", "CSF.pct"].iloc[0]
alpha = 0.05
max_cells = 80

rows = []

for n in range(1, max_cells + 1):
    for k in range(0, n + 1):
        purity = k / n

        # Binomial test under null
        pval = binomtest(k, n, p_csf, alternative="two-sided").pvalue

        if pval < alpha:
            rows.append({
                "n_cells": n,
                "purity": purity
            })

sig_df = pd.DataFrame(rows)
print(sig_df[sig_df["n_cells"] == 12])

#Prepare the figure
plt.figure(figsize=(4, 3.75))

#Plot data points
plt.scatter(
    sig_df["n_cells"],
    sig_df["purity"],
    s=10,
    color="black"
)

plt.axhline(0.55, linestyle="--", linewidth=1)
plt.axvline(20, linestyle="--", linewidth=1)
plt.xlabel("Number of cells discovered")
plt.ylabel("Fraction of cells in CSF of a given clone")
plt.title("MS only")

plt.tight_layout()
plt.show()

In [ ]:
## Given these statistics, make permutations to indicate what is highly significant
# Parameters
p_csf = compart_sum_grp.loc[compart_sum_grp["catpat"] == "NIC", "CSF.pct"].iloc[0]
alpha = 0.05
max_cells = 80

rows = []

for n in range(1, max_cells + 1):
    for k in range(0, n + 1):
        purity = k / n

        # Binomial test under null
        pval = binomtest(k, n, p_csf, alternative="two-sided").pvalue

        if pval < alpha:
            rows.append({
                "n_cells": n,
                "purity": purity
            })

sig_df = pd.DataFrame(rows)
print(sig_df[sig_df["n_cells"] == 12])

#Prepare the figure
plt.figure(figsize=(4, 3.75))

#Plot data points
plt.scatter(
    sig_df["n_cells"],
    sig_df["purity"],
    s=10,
    color="black"
)

plt.axhline(0.25, linestyle="--", linewidth=1)
plt.axvline(20, linestyle="--", linewidth=1)
plt.xlabel("Number of cells discovered")
plt.ylabel("Fraction of cells")
plt.title("NIC only")

plt.tight_layout()
plt.show()

# Statistical thresholds for Kavaka et al 

In [ ]:
########################################
##  CSF and PB fractions are from R   ##
########################################

# Parameters fitting Kavaka et al 2023
p_csf = 0.04744812  
alpha = 0.05
max_cells = 80

rows = []

for n in range(1, max_cells + 1):
    for k in range(0, n + 1):
        purity = k / n

        # Binomial test under null
        pval = binomtest(k, n, p_csf, alternative="two-sided").pvalue

        if pval <= alpha:
            rows.append({
                "n_cells": n,
                "purity": purity
            })

sig_df = pd.DataFrame(rows)

In [ ]:
sig_df[sig_df["n_cells"] == 13]

In [ ]:
# Prepare the figure
fig, ax = plt.subplots(figsize=(4, 3.75))

# Plot data points
ax.scatter(
    sig_df["n_cells"],
    sig_df["purity"],
    s=10,
    color="black"
)

ax.axhline(0.230769, linestyle="--", linewidth=1)
ax.axvline(13, linestyle="--", linewidth=1)

ax.set_xlabel("Number of cells discovered")
ax.set_ylabel("Fraction of cells in CSF of a given clone")
ax.set_title("Aggregated MS & Ctrl")

fig.tight_layout()

# ---- Save BEFORE show ----
fig.savefig(
    "../../05_plots/13_final_plots/27_clone_size_vs_csf_fraction_kavaka.pdf",
    bbox_inches="tight",
    format="pdf",
    dpi=400
)

plt.show()

# Barplots to visualize CSF to PB-biased compartment distributions

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 1: prepare the data
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Get data
obs_df = sdata["gex"].obs.copy()

# Make clone_handle categorical
obs_df["clone_handle"] = obs_df["clone_handle"].astype("category")

# Filter for clone_handle in comp_set and remove donors Pt4 and Pt5
obs_df = obs_df[obs_df["total_clone_count"] > 12]
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()
obs_df["clone_handle"] = obs_df["clone_handle"].cat.remove_unused_categories()

#Polish dataframe
print("I have this many unique clones: ", obs_df["clone_handle"].nunique())
obs_df = obs_df[["donor", "catpat", "clone_handle", "Compartment", "total_clone_count", "type_x_clone_count"]].drop_duplicates()

# Report
print(obs_df[["clone_handle", "catpat"]].drop_duplicates().groupby(["catpat"]).size())

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 2: Compute Compartment Percentages
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Compute % contribution of each compartment per clone
obs_df["Compartment_pct"] = 100 * obs_df["type_x_clone_count"] / obs_df["total_clone_count"]

# Create input dataframe for pivoting
df_input = obs_df[["clone_handle", "Compartment", "Compartment_pct", "catpat"]]

# Pivot the data so you have clone_handle and catpat as indexes with 2 columns "CSF" and PB"
pivot_table_comp = df_input.pivot_table(index=["clone_handle", "catpat"], columns="Compartment", values="Compartment_pct", fill_value=0)

# Rename columns
pivot_table_comp.columns = [f"{col}.pct" for col in pivot_table_comp.columns]

# Sort to show CSF-high clones first
pivot_table_comp = pivot_table_comp.sort_values(by="CSF.pct", ascending=False)  #Stores the compartment-wise dataset for plotting
clone_order = pivot_table_comp.index.tolist() #Will be used for L3_ann plots later
num_bars_comp = len(pivot_table_comp)

#Clone order for PB and CSF individually - these indexes are combinations of clone_handle and catpat (!!): 
pivot_table_comp.sort_values(by="CSF.pct", ascending=False)
print(pivot_table_comp.head(2))
clone_order_pb = pivot_table_comp[pivot_table_comp["CSF.pct"] != 100].sort_values(by="CSF.pct", ascending=False).reset_index()["clone_handle"].tolist()  #Stores the compartment-wise dataset for plotting
clone_order_csf = pivot_table_comp[pivot_table_comp["PB.pct"] != 100].sort_values(by="CSF.pct", ascending=False).reset_index()["clone_handle"].tolist()   #Stores the compartment-wise dataset for plotting

#Save pivot_table_comp for later reference: 
pivot_table_comp.to_csv("../../06_csv_outputs/Clones/2026_7_CD8_hyperexpanded_clone_handles_and_composition_table_across_compartments.csv")
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 3: plot NAKED CSF to PB transition
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Ensure index is reset
pivot_table_comp = pivot_table_comp.reset_index()

# Setup
catpats = ["NIC", "MS"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)

if n_facets == 1:
    axes = [axes]  # Ensure list in single facet case

# Plot per facet
for i, cat in enumerate(catpats):
    ax = axes[i]
    df_cat = pivot_table_comp[pivot_table_comp["catpat"] == cat].reset_index(drop=True)

    #Add a horizontal line that represent the mean %CSF of cells
    ax.axhline(y=53.8, color='black', linestyle='solid', linewidth=0.5)

    x = np.arange(len(df_cat))
    width = 1

    ax.bar(x, df_cat["CSF.pct"].values, width, color="#4B8AC1", edgecolor="none")
    ax.bar(x, df_cat["PB.pct"].values, width, bottom=df_cat["CSF.pct"].values, color="#C06A6A", edgecolor="none")
    
    # Clean up styling
    ax.set_xticks([])
    ax.tick_params(axis='y', labelsize=9)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f"{cat} clones")

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    ax.axvline(-0.5, color="black", linewidth=1)

# Save figure
fig.savefig("../../05_plots/13_final_plots/8_CD8_PB_to_CSF_percentage_transition_hyperexpanded.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()

## Save the clone handles so that you can easily plot the exact clones as used here
list_of_clones_expansion = obs_df["clone_handle"].drop_duplicates().tolist()
print("Length of saved list: ", len(list_of_clones_expansion))

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 4: Prepare L3_cluster data for PB
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Get data
obs_df = sdata["gex"].obs.copy()

# Make clone_handle categorical
obs_df["clone_handle"] = obs_df["clone_handle"].astype("category")

## Filter
#Use these exact same clones as in CSF to PB transition plot.
obs_df = obs_df[obs_df["clone_handle"].isin(list_of_clones_expansion)]
obs_df = obs_df[obs_df["Compartment"]== "PB"]

# Polish categoricals
obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
obs_df["clone_handle"] = obs_df["clone_handle"].cat.remove_unused_categories()
obs_df["Compartment"] = obs_df["Compartment"].cat.remove_unused_categories()
obs_df["L3_ann"] = obs_df["L3_ann"].cat.remove_unused_categories()

# select necessary columns
obs_df = obs_df[["catpat", "clone_handle", "L3_ann"]]

# Group by clone_handle, L3_ann and catpat and count the number of cells in the single-cell object
L3_counts = (
    obs_df
    .groupby(["clone_handle", "L3_ann", "catpat"])
    .size()
    .reset_index(name="count")
)

# Pivot count data to generate a wide format - this generates many rows full with 0 values. 
count_tableL3 = L3_counts[["clone_handle",  "catpat", "L3_ann", "count"]].drop_duplicates().pivot(index=["clone_handle", "catpat"], columns=["L3_ann"], values="count").fillna(0)

# Remove those without any counts
### Take all columns except clone_handle and catpat
cluster_columns = count_tableL3.columns.difference(["clone_handle", "catpat"])
count_tableL3 = count_tableL3.loc[(count_tableL3[cluster_columns].sum(axis=1) > 0)]

# Generate the percentage based wide dataframe
count_tableL3_pct = (
    count_tableL3.div(count_tableL3.sum(axis=1), axis=0)
    .fillna(0)
    .mul(100)
)

## Add _pct
count_tableL3_pct = count_tableL3_pct.add_suffix("_pct")

# Reset the index
count_tableL3_pct = count_tableL3_pct.reset_index()

### ----
## Before plotting reorder the dataframe
### ----
count_tableL3_pct = count_tableL3_pct.set_index("clone_handle")
count_tableL3_pct = count_tableL3_pct.loc[clone_order_pb]
count_tableL3_pct = count_tableL3_pct.reset_index()

print(len(count_tableL3_pct))

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure : NAKED plot of L3_ann clusters from PB ---
# %%%%%%%%%%%%%%%%%%%%%

#reset index to unstack the multiindex: 
pivot_table_L3_pb_plot = count_tableL3_pct

# Set up colors
l3_color_dict_pb = L3_ann_colors

# Create one subplot per category (MS, NIC)
catpats = ["MS", "NIC"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)

if n_facets == 1:
    axes = [axes]  # Ensure it's always a list

# Plot each group
for i, cat in enumerate(catpats):
    ax = axes[i]
    
    df_cat = pivot_table_L3_pb_plot[pivot_table_L3_pb_plot["catpat"] == cat].reset_index(drop=True)
    x = np.arange(len(df_cat))
    bottom_vals = np.zeros(len(df_cat))
    
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        color = l3_color_dict_pb.get(l3_cluster.replace("_pct", ""), "#333333")
        y_vals = df_cat[l3_cluster].values
        ax.bar(x, y_vals, width=1, bottom=bottom_vals, label=l3_cluster.replace("_pct", ""), color=color, edgecolor="none")
        bottom_vals += y_vals

    ax.set_title(f"{cat} clones\nL3 cluster contributions ({len(df_cat)} clones)", fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    # ax.set_yticks([])
    ax.tick_params(axis='y', labelsize=9)
    ax.axvline(-0.5, color="black", linewidth=1)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# Save the figure
fig.savefig("../../05_plots/13_final_plots/9_CD8_PB_LOESS_trends_by_L3_cluster_facet_catpat_NAKED.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()



# %%%%%%%%%%%%%%%%%%%%%
# --- Figure:  LOESS regression of PB ---
# %%%%%%%%%%%%%%%%%%%%%

# Faceting setup
catpats = ["MS", "NIC"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)
if n_facets == 1:
    axes = [axes]

for i, cat in enumerate(catpats):
    ax = axes[i]
    
    # Subset data
    df_cat = pivot_table_L3_pb_plot[pivot_table_L3_pb_plot["catpat"] == cat].reset_index(drop=True)
    x_vals = np.arange(len(df_cat))

    # LOWESS plot per L3 cluster
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        cluster_label = l3_cluster.replace("_pct", "")
        y_vals = df_cat[l3_cluster].values
        smoothed = lowess(y_vals, x_vals, frac=0.35)
        color = l3_color_dict_pb.get(cluster_label, "#333333")
        ax.plot(smoothed[:, 0], smoothed[:, 1], label=cluster_label, color=color, linewidth=2)

    # Axes styling
    ax.set_xticks([])
    ax.set_yticks([0, 20, 40, 60, 80])
    ax.set_ylim(0, 90)
    ax.set_xlabel("")
    ax.set_title(f"{cat} clones", fontsize=11)

    if i == 0:
        ax.set_ylabel("% of cells in cluster", fontsize=10)
    else:
        ax.set_ylabel("")

    ax.tick_params(axis='y', labelsize=9)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    ax.axvline(-0.5, color="black", linewidth=1)

# Save the faceted LOWESS figure
fig.savefig("../../05_plots/13_final_plots/11_CD8_PB_LOESS_trends_by_L3_cluster_facet_catpat.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 4: Prepare L3_cluster data for PB
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Get data
obs_df = sdata["gex"].obs.copy()

# Make clone_handle categorical
obs_df["clone_handle"] = obs_df["clone_handle"].astype("category")

## Filter
#Use these exact same clones as in CSF to PB transition plot.
obs_df = obs_df[obs_df["clone_handle"].isin(list_of_clones_expansion)]
obs_df = obs_df[obs_df["Compartment"]== "CSF"]

# Polish categoricals
obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
obs_df["clone_handle"] = obs_df["clone_handle"].cat.remove_unused_categories()
obs_df["Compartment"] = obs_df["Compartment"].cat.remove_unused_categories()
obs_df["L3_ann"] = obs_df["L3_ann"].cat.remove_unused_categories()

# select necessary columns
obs_df = obs_df[["catpat", "clone_handle", "L3_ann"]]

# Group by clone_handle, L3_ann and catpat and count the number of cells in the single-cell object
L3_counts = (
    obs_df
    .groupby(["clone_handle", "L3_ann", "catpat"])
    .size()
    .reset_index(name="count")
)

# Pivot count data to generate a wide format - this generates many rows full with 0 values. 
count_tableL3 = L3_counts[["clone_handle",  "catpat", "L3_ann", "count"]].drop_duplicates().pivot(index=["clone_handle", "catpat"], columns=["L3_ann"], values="count").fillna(0)

# Remove those without any counts
### Take all columns except clone_handle and catpat
cluster_columns = count_tableL3.columns.difference(["clone_handle", "catpat"])
count_tableL3 = count_tableL3.loc[(count_tableL3[cluster_columns].sum(axis=1) > 0)]

# Generate the percentage based wide dataframe
count_tableL3_pct = (
    count_tableL3.div(count_tableL3.sum(axis=1), axis=0)
    .fillna(0)
    .mul(100)
)

## Add _pct
count_tableL3_pct = count_tableL3_pct.add_suffix("_pct")

# Reset the index
count_tableL3_pct = count_tableL3_pct.reset_index()

### ----
## Before plotting reorder the dataframe
### ----
count_tableL3_pct = count_tableL3_pct.set_index("clone_handle")
count_tableL3_pct = count_tableL3_pct.loc[clone_order_csf]
count_tableL3_pct = count_tableL3_pct.reset_index()

print(len(count_tableL3_pct))

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure : NAKED plot of L3_ann clusters from csf ---
# %%%%%%%%%%%%%%%%%%%%%

#reset index to unstack the multiindex: 
pivot_table_L3_csf_plot = count_tableL3_pct

# Set up colors
l3_color_dict_csf = L3_ann_colors

# Create one subplot per category (MS, NIC)
catpats = ["MS", "NIC"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)

if n_facets == 1:
    axes = [axes]  # Ensure it's always a list

# Plot each group
for i, cat in enumerate(catpats):
    ax = axes[i]
    
    df_cat = pivot_table_L3_csf_plot[pivot_table_L3_csf_plot["catpat"] == cat].reset_index(drop=True)
    x = np.arange(len(df_cat))
    bottom_vals = np.zeros(len(df_cat))
    
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        color = l3_color_dict_csf.get(l3_cluster.replace("_pct", ""), "#333333")
        y_vals = df_cat[l3_cluster].values
        ax.bar(x, y_vals, width=1, bottom=bottom_vals, label=l3_cluster.replace("_pct", ""), color=color, edgecolor="none")
        bottom_vals += y_vals

    ax.set_title(f"{cat} clones\nL3 cluster contributions ({len(df_cat)} clones)", fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([])
    # ax.set_yticks([])
    ax.tick_params(axis='y', labelsize=9)
    ax.axvline(-0.5, color="black", linewidth=1)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# Save the figure
fig.savefig("../../05_plots/13_final_plots/9_CD8_CSF_LOESS_trends_by_L3_cluster_facet_catpat_NAKED.pdf", bbox_inches='tight', format="pdf", dpi=400)
plt.show()



# %%%%%%%%%%%%%%%%%%%%%
# --- Figure:  LOESS regression of CSF ---
# %%%%%%%%%%%%%%%%%%%%%

# Faceting setup
catpats = ["MS", "NIC"]
n_facets = len(catpats)

fig, axes = plt.subplots(1, n_facets, figsize=(6 * n_facets, 3), constrained_layout=True, sharey=True)
if n_facets == 1:
    axes = [axes]

for i, cat in enumerate(catpats):
    ax = axes[i]
    
    # Subset data
    df_cat = pivot_table_L3_csf_plot[pivot_table_L3_csf_plot["catpat"] == cat].reset_index(drop=True)
    x_vals = np.arange(len(df_cat))

    # LOWESS plot per L3 cluster
    for l3_cluster in df_cat.columns:
        if l3_cluster in ["clone_handle", "catpat"]:
            continue
        cluster_label = l3_cluster.replace("_pct", "")
        y_vals = df_cat[l3_cluster].values
        smoothed = lowess(y_vals, x_vals, frac=0.35)
        color = l3_color_dict_csf.get(cluster_label, "#333333")
        ax.plot(smoothed[:, 0], smoothed[:, 1], label=cluster_label, color=color, linewidth=2)

    # Axes styling
    ax.set_xticks([])
    ax.set_yticks([0, 20, 40, 60, 80])
    ax.set_ylim(0, 100)
    ax.set_xlabel("")
    ax.set_title(f"{cat} clones", fontsize=11)

    if i == 0:
        ax.set_ylabel("% of cells in cluster", fontsize=10)
    else:
        ax.set_ylabel("")

    ax.tick_params(axis='y', labelsize=9)

    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    ax.axvline(-0.5, color="black", linewidth=1)

# Save the faceted LOWESS figure
fig.savefig("../../05_plots/13_final_plots/11_CD8_csf_LOESS_trends_by_L3_cluster_facet_catpat.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

## Statistic for CSF-biased clones

In [ ]:
## Number of CD8 T cells discovered in PB and CSF is similar
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[~obs_df["clone_handle"].isna()]

## annotate CSF-biased clones per donor
count_df = obs_df[["donor", "catpat", "clone_handle", "Compartment"]].value_counts().reset_index()
count_df = count_df.pivot_table(index = ["donor", "catpat", "clone_handle"], columns = "Compartment", values="count", fill_value=0).reset_index()
count_df["total"] = count_df["CSF"] + count_df["PB"]
count_df = count_df[count_df["total"]>12]
count_df["fra_csf"] = count_df["CSF"] / count_df["total"]
count_df["biased_ann"] = np.where(count_df["fra_csf"]> 0.538, "CSF-biased", "Other")

## enumerate the number of CSF-biased clones per donor
sum_df = count_df[["donor", "catpat", "biased_ann"]].value_counts().reset_index().pivot_table(index=["donor", "catpat"], columns="biased_ann", values = "count", fill_value = 0).reset_index()
sum_df["pct_csfbiased"] = 100*sum_df["CSF-biased"] / (sum_df["CSF-biased"] + sum_df["Other"])

#Remove all unused categories
for col in sum_df.select_dtypes(["category"]):
    sum_df[col] = sum_df[col].cat.remove_unused_categories()

fig, ax = plt.subplots(figsize=(6.5,5))

# Boxplot (keep white boxes)
# Define manual colors
palette = {
    "NIC": "#FFFFFF",   # blue
    "MS": "#000000"       # red
}

sb.boxplot(
    data=sum_df,
    x="catpat",
    y="pct_csfbiased",
    hue="catpat",
    log_scale=False,
    showcaps=True,
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette=palette,
    ax=ax
)

# Stripplot (colored dots)
sb.stripplot(
    data=sum_df,
    x="catpat",
    y="pct_csfbiased",
    hue="catpat",
    log_scale=False,
    s=10,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette=palette, 
    ax=ax
)

# ## Statistics
# MS_vals = plot_df[plot_df["catpat"] == "MS"]["count_per_ml"]
# ctrl_vals = plot_df[plot_df["catpat"] == "NIC"]["count_per_ml"]
# stat, pval = mannwhitneyu(MS_vals, ctrl_vals, alternative='two-sided')
# pval

## Statistics on CD8 count per ml CSF

In [ ]:
## Define the volumne of CSF taken per donor
csf_ml = {
    "Pt1" : 20,
    # "Pt4" : 11, 
    # "Pt5" : 8, 
    "Pt8" : 10, 
    "Pt11" : 15, 
    "Pt12" : 15, 
    "Pt14" : 14, 
    "Pt17" : 14, 
    "Pt18" : 12, 
    "Pt19" : 14, 
    "Pt20" : 14.5, 
    "Pt21" : 14
}


In [ ]:
## Number of CD8 T cells discovered in PB and CSF is similar
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[~obs_df["clone_handle"].isna()]

# Add a new column
obs_df["Puncture_amount"] = obs_df["donor"].map(csf_ml)

#Remove all unused categories
for col in obs_df.select_dtypes(["category"]):
    obs_df[col] = obs_df[col].cat.remove_unused_categories()

#Count
plot_df = obs_df[["donor", "Compartment", "catpat", "Puncture_amount"]].value_counts().reset_index()

#Normalize
plot_df["count_norm"] = plot_df["count"] / plot_df["Puncture_amount"]

fig, ax = plt.subplots(figsize=(3, 3))

# Define colors for compartments
# compartment_colors = {"CSF": "#4B8AC1", "PB": "#C06A6A"}

sb.boxplot(
    data=plot_df,
    x="Compartment",
    y="count_norm",
    hue="catpat",
    boxprops={'facecolor': 'white'},
    showfliers=False,
    palette={"NIC": "#FFFFFF", "MS": "#000000"},
    ax=ax
)

sb.stripplot(
    data=plot_df,
    x="Compartment",
    y="count_norm",
    hue="catpat",
    dodge=True,
    s=8,
    jitter=True,
    alpha=0.8,
    linewidth=0.5,
    edgecolor="black",
    palette={"NIC": "#FFFFFF", "MS": "#000000"},
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("Number of CD8 T cells/ml CSF")

# Remove duplicated legends caused by boxplot + stripplot
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:len(palette)], labels[:len(palette)])

sb.despine()

plt.tight_layout()
plt.show()


In [ ]:
# --- Prepare obs_df ---
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df = obs_df[obs_df["clone_size"] > 20]
obs_df = obs_df[obs_df["Compartment"] == "PB"]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 


# Leiden cluster
cls = "leiden_1.8"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

# --- Statistics: Mann–Whitney U per cluster ---
def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "RRMS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"] == "Control", "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided")
        pvals[cl] = p
    else:
        pvals[cl] = np.nan

# --- Plot ---
plt.figure(figsize=(10, 4))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    showfliers=False,
    linewidth=1
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    jitter=0.2,
    alpha=0.5,
    size=5,
    linewidth=0,
    ax=ax
)

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], labels[:2], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage")
plt.xticks(rotation=45, ha="right")

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = pvals.get(cl, np.nan)
    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08
    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )
    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

plt.tight_layout()
plt.show()

pvals_df = (
    pd.DataFrame.from_dict(pvals, orient="index", columns=["p_value"])
    .reset_index()
    .rename(columns={"index": cls})
)
print(pvals_df)


## Barplots + CSF/PB-biases + DEG analysis

In [ ]:
sum_df = sdata["gex"].obs.copy()
# sum_df = sum_df[~sum_df["L3_ann"].str.contains("Tn", regex=True)]

sum_df = sum_df[["donor", "Compartment"]]

# Count number of unique Compartment categories per donor
counts_per_donor = sum_df.groupby(["donor", "Compartment"]).size().reset_index().pivot(index="donor", columns="Compartment", values=0)

# % CSF
counts_per_donor["total"] = counts_per_donor["PB"] + counts_per_donor["CSF"]
counts_per_donor["CSF_pct"] = 100*counts_per_donor["CSF"] / counts_per_donor["total"]

# Calculate mean and std
mean_values = counts_per_donor.mean()
std_values = counts_per_donor.std()

# Combine into a single DataFrame
summary = pd.DataFrame({
    "mean": mean_values,
    "std": std_values
})

print("Kilian's threshold = mean % of CSF occupance per donor")
print(summary)

In [ ]:
sdata["gex"].obs.head(3)

In [ ]:
# %%%%%%%%%%%
# Selecting statistically significant clones
# %%%%%%%%%%%

# print(sdata["gex"].obs["Sig_label"].drop_duplicates())

# All nan values are clone size == 0
sdata["gex"].obs[sdata["gex"].obs["Sig_label"].isna()].sort_values("clone_size", ascending=False).head(3)

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 1: prepare the data
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Get data
obs_df = sdata["gex"].obs.copy()

# Make clone_handle categorical
obs_df["clone_handle"] = obs_df["clone_handle"].astype("category")

# Filter for clone_handle in comp_set and remove donors Pt4 and Pt5
obs_df = obs_df[obs_df["total_clone_count"] > 12]
obs_df = obs_df[~obs_df["donor"].isin(["Pt4", "Pt5"])]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories()
obs_df["catpat"] = obs_df["catpat"].cat.remove_unused_categories()
obs_df["clone_handle"] = obs_df["clone_handle"].cat.remove_unused_categories()

#Polish dataframe
print("I have this many unique clones: ", obs_df["clone_handle"].nunique())
obs_df = obs_df[["donor", "catpat", "clone_handle", "Compartment", "total_clone_count", "type_x_clone_count"]].drop_duplicates()

# Report
print(obs_df[["clone_handle", "catpat"]].drop_duplicates().groupby(["catpat"]).size())

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Step 2: Compute Compartment Percentages
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Compute % contribution of each compartment per clone
obs_df["Compartment_pct"] = 100 * obs_df["type_x_clone_count"] / obs_df["total_clone_count"]

# Create input dataframe for pivoting
df_input = obs_df[["clone_handle", "Compartment", "Compartment_pct", "catpat"]]

# Pivot the data so you have clone_handle and catpat as indexes with 2 columns "CSF" and PB"
pivot_table_comp = df_input.pivot_table(index=["clone_handle", "catpat"], columns="Compartment", values="Compartment_pct", fill_value=0)

# Rename columns
pivot_table_comp.columns = [f"{col}.pct" for col in pivot_table_comp.columns]

# Sort to show CSF-high clones first
pivot_table_comp = pivot_table_comp.sort_values(by="CSF.pct", ascending=False)  #Stores the compartment-wise dataset for plotting
clone_order = pivot_table_comp.index.tolist() #Will be used for L3_ann plots later
num_bars_comp = len(pivot_table_comp)

#Clone order for PB and CSF individually - these indexes are combinations of clone_handle and catpat (!!): 
pivot_table_comp.sort_values(by="CSF.pct", ascending=False)
print(pivot_table_comp.head(2))
clone_order_pb = pivot_table_comp[pivot_table_comp["CSF.pct"] != 100].sort_values(by="CSF.pct", ascending=False).reset_index()["clone_handle"].tolist()  #Stores the compartment-wise dataset for plotting
clone_order_csf = pivot_table_comp[pivot_table_comp["PB.pct"] != 100].sort_values(by="CSF.pct", ascending=False).reset_index()["clone_handle"].tolist()   #Stores the compartment-wise dataset for plotting

# %%%%%%%%%%%%%%%%%%%
# Define highlight clone sets
# %%%%%%%%%%%%%%%%%%
CSF14_clones = pivot_table_comp[pivot_table_comp["CSF.pct"] >= 53.8].sort_values(by="CSF.pct", ascending=False).reset_index()["clone_handle"].tolist()  #Stores the compartment-wise dataset for plotting
PB14_clones = pivot_table_comp[pivot_table_comp["PB.pct"] == 100].sort_values(by="CSF.pct", ascending=False).reset_index()["clone_handle"].tolist()  #Stores the compartment-wise dataset for plotting
highlight_clones = set(CSF14_clones + PB14_clones)
print(len(PB14_clones), " PB-biased clones, and this many ", len(CSF14_clones), "CSF-biased clones")

# Export these clones: 
CSF14_clones_series = pd.Series(CSF14_clones)
CSF14_clones_series.to_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_CSF14_clonehandles_ForCSF.csv", index=False)

PB14_clones_series = pd.Series(PB14_clones)
PB14_clones_series.to_csv(r"..\..\06_csv_outputs\Clones\2025_6_CD8_PB14_clonehandles_ForCSF.csv", index=False)

# Extract L3_ann color dictionary
l3_labels = sdata_highfrequent["gex"].obs["L3_ann"].cat.categories
l3_colors = sdata_highfrequent["gex"].uns["L3_ann_colors"]
l3_color_dict = dict(zip(l3_labels, l3_colors))

# %%%%%%%%%%%%%%%%%%%%%
# --- Figure: clones selected for differential gene expression
# %%%%%%%%%%%%%%%%%%%%%

# Set up subplots
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 4), constrained_layout=True)
x = np.arange(len(pivot_table_comp))
width = 1

clone_handles = pivot_table_comp.index.get_level_values("clone_handle")
csf_colors = ["#4B8AC1" if clone in highlight_clones else "#E0E0E0" for clone in clone_handles]
pb_colors = ["#C06A6A" if clone in highlight_clones else "#D8D8D8" for clone in clone_handles]

ax1.bar(x, pivot_table_comp["CSF.pct"], width, label="CSF", color=csf_colors, edgecolor="none")  #Here I replace color="skyblue" with a list of colors according to the number of "top" clones were selected
ax1.bar(x, pivot_table_comp["PB.pct"], width, bottom=pivot_table_comp["CSF.pct"], label="PB", color=pb_colors, edgecolor="none") #Here I replace color="#C06A6A" with a list of colors according to the number of "top" clones were selected

#Add a horizontal line that represent the significance threshold for enrichment in CSF
ax1.axhline(y=53.8)  #This corresponds to the signif threshold form earlier

#Polish the plot
ax1.set_title(f"Clones w. more than 12 observed cells in total\n"
              f"CSF vs PB composition per clone\n{len(pivot_table_comp)} clones", fontsize=12)
ax1.set_xlabel("Clone Handle")
ax1.set_ylabel("Percentage")
ax1.set_xticks([])
for spine in ["top", "right", "left", "bottom"]:
    ax1.spines[spine].set_visible(False)
ax1.axvline(-0.5, color="black", linewidth=1)
ax1.legend(title="Compartment", bbox_to_anchor=(1.01, 1), loc="upper left", borderaxespad=0)

# Save the figure (optional)
fig1.savefig("../../05_plots/13_final_plots/14_CD8_CSF_PB_stackbar_highlighted.pdf", bbox_inches='tight', format="pdf", dpi=400)


In [ ]:
# Set figure params
sc.settings.set_figure_params(figsize=(4.3, 4))

# Define highlight clone sets
CSF14_clones 
PB14_clones

#For visualization
sdata["gex"].obs["top_vs_bot_clones2"] = pd.Categorical(
                      np.where(sdata["gex"].obs["clone_handle"].isin(PB14_clones), "PB-biased", 
                                np.where((sdata["gex"].obs["clone_handle"].isin(CSF14_clones)) & (sdata["gex"].obs["Compartment"] == "PB"), "CSF-biased in PB", 
                                         np.where(sdata["gex"].obs["clone_handle"].isin(CSF14_clones), "CSF-biased in CSF", "unselected"))))

#For later DEGs
sdata["gex"].obs["top_vs_bot_clones"] = pd.Categorical(
                      np.where(sdata["gex"].obs["clone_handle"].isin(PB14_clones), "PB-biased", 
                                np.where(sdata["gex"].obs["clone_handle"].isin(CSF14_clones), "CSF-biased",  "unselected")))


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# UMAP Plot
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# Set figure parameters
sc.settings.set_figure_params() #Back to default

# Generate the UMAP figure and capture it
fig = sc.pl.umap(
    sdata["gex"],
    color=["top_vs_bot_clones2"],
    groups=["CSF-biased in PB", "CSF-biased in CSF", "PB-biased"],
    palette={"CSF-biased in CSF" : "#4B8AC1",  "CSF-biased in PB" : "#FD8D3C", "PB-biased" : "#C06A6A",  "unselected":"lightgrey"},
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    size=40,
    return_fig=True
)

# Save the figure
fig.savefig("../../05_plots/13_final_plots/16_showallcells_CD8_CSF_PB_UMAP_top_vs_bottom_clones.pdf", bbox_inches='tight', format="pdf", dpi = 400)

plt.show()



# Differential gene expression analysis
## CSF biased vs PB biased

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
sdata_pb = sdata.copy()
mu.pp.filter_obs(sdata_pb["gex"], "Compartment", lambda x: (x == "PB")) 

## Define the filtered to alter: 
sdata_pb_filtered = sdata_pb["gex"].copy()

# --- Filter genes before differential gene expression ---
## Step 1: Generate lists of expressed genes (>X%) for each of the 3 populations of observed cells T cells, B cells and interacting cells

## --- Expression in CSF-biased T cells
adata = sdata_pb["gex"][sdata_pb["gex"].obs["top_vs_bot_clones"] == "CSF-biased"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

## Report /lookup
print(fraction_df[fraction_df["names"] == "PDCD1"])

## --- Expression in PB-biased T cells
adata = sdata_pb["gex"][sdata_pb["gex"].obs["top_vs_bot_clones"] == "PB-biased"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

## Report /lookup
print(fraction_df[fraction_df["names"] == "PDCD1"])

#The total set of genes
gene_set_expressed = set(gene_list_10pct_1+gene_list_10pct_2)
print(len(gene_set_expressed))

# -----------------------------
# Step 2: exclude unwanted genes
# -----------------------------
genes = sdata_pb["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))
exclude_genes = set(genes[exclude_mask])
genes_to_keep = sorted(gene_set_expressed - exclude_genes)
print("Final genes:", len(genes_to_keep))

# -----------------------------
# Step 3: subset AnnData correctly
# -----------------------------
sdata_pb_filtered = sdata_pb_filtered[:, genes_to_keep].copy()

# Perform naive differential gene expression
sc.tl.rank_genes_groups(sdata_pb_filtered, groupby="top_vs_bot_clones", groups=["CSF-biased"], reference="PB-biased", key_added="CSF_biased_gene_set", method="wilcoxon")
sc.tl.rank_genes_groups(sdata_pb_filtered, groupby="top_vs_bot_clones", groups=["PB-biased"], reference="CSF-biased", key_added="PB_biased_gene_set", method="wilcoxon")

In [ ]:
# Display initial results
sc.pl.rank_genes_groups_dotplot(
     sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=30, key="CSF_biased_gene_set")
sc.pl.rank_genes_groups_dotplot(
     sdata_pb_filtered, groupby="top_vs_bot_clones", standard_scale="var", n_genes=30, key="PB_biased_gene_set")


In [ ]:
# %%% 
# Vulcano plot function
# %%%

# # From: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
LOG_FOLD_CHANGE = 0.5
PVALUE = -np.log10(0.05)  #Higher than p_adj 0.05. 
Add_labels = []
Add_labels = ['GZMK', 'CCR7', "GAPDH", "CCR5", 'CD27', "CD28", 'GPR183', 'CXCR4', "CXCR3", "CCR5", "MS4A1", "AC004448.2", "CLDND1", "CCL5", "KIR2DL4", "KIR2DL1", "PASK", "DKK3", "COTL1", "SARAF", "PPP1R14B", "NKG7", 
              'GZMB', 'ZNF683', 'GZMH', 'KLRD1',  'CX3CR1', 'FNDC3B',  'KLRC2', "TYROBP", "AOAH", "GBP5", "S100A4", "EFHD1", "HLA-C", "EFHD2", "MYL12A", "ST8SIA1", "ITGB1", "LGALS1", "GNLY", "CD52", "IL32", "IRF1", "HOPX", 
              "ZEB2", "FCRL6", "PRF1", "NKG7", "DUSP2", "IL7R", "EEF1A1", "SARAF", "ZFP36", "NELL2", "PDE4B", "PHACTR2" ]

VOLCANO_PALETTE = {
    "neg": "#D89A52",   # e.g. CSF-biased (negative logFC)
    "pos": "#C06A6A",   # e.g. PB-biased (positive logFC)
    "ns":  "lightgrey"
}

def volcano_plot(adata, group_key, title=None, palette=VOLCANO_PALETTE):
    result = sc.get.rank_genes_groups_df(
        adata, group=None, key=group_key
    ).copy()

    #Lookup
    print(result[result["names"] == "PDCD1"])

    result["-log_padj"] = -np.log10(result["pvals_adj"].astype(float))

    #Remove dots with a very low log2fc
    result=result[(result["logfoldchanges"] > 0.25) | (result["logfoldchanges"] < -0.25)]

    #Drop stuff that is na
    result = result.dropna(subset="names")

    # Significant vs non-significant
    sig = result.loc[
        (abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) &
        (result["-log_padj"] > PVALUE)
    ]

    nonsig = result.loc[
        (abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) |
        (result["-log_padj"] <= PVALUE)
    ]

    # Split significant genes by direction
    sig_pos = sig.loc[sig["logfoldchanges"] > 0]
    sig_neg = sig.loc[sig["logfoldchanges"] < 0]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.grid(True, zorder=0)

    # Non-significant genes
    sb.regplot(
        x=nonsig["logfoldchanges"],
        y=nonsig["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "alpha": 0.5, "zorder": 2},
        ax=ax,
        color=palette["ns"]
    )

    # Positively enriched
    sb.regplot(
        x=sig_pos["logfoldchanges"],
        y=sig_pos["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["pos"],
        label="Positive enrichment"
    )

    # Negatively enriched
    sb.regplot(
        x=sig_neg["logfoldchanges"],
        y=sig_neg["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 60, "zorder": 3},
        ax=ax,
        color=palette["neg"],
        label="Negative enrichment"
    )

    # Rasterize only the dots
    # for coll in ax.collections:
    #     coll.set_rasterized(True)

    # Gene labels
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        ax.text(
            row["logfoldchanges"],
            row["-log_padj"],
            row["names"],
            fontsize=8,
            ha="right",
            va="bottom",
            zorder=4
        )

    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log padj")

    #Set limits
    ax.set_xlim(-6, 6)

    if title is None:
        title = group_key.replace("_", " ")
    ax.set_title(title)

    ax.legend(frameon=False)
    sb.despine()
    ax.grid(False)
    plt.tight_layout()

    plt.savefig(
        "../../05_plots/13_final_plots/17_CD8_vulcanoPlot_CSFvPB_bias.pdf",
        bbox_inches="tight",
        format="pdf",
        dpi=400
    )
    plt.show()

volcano_plot(sdata_pb_filtered, group_key="PB_biased_gene_set", title="<-- CSF-biased & PB-biased -->") 

# Export CSF/PB-biased DEGs

In [ ]:
datPBbiased = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None, key ="PB_biased_gene_set").dropna(subset="names")
datPBbiased.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_06_CD8_CSFnPB_biased_DEGs.csv")

datCSFbiased = sc.get.rank_genes_groups_df(sdata_pb_filtered, group = None, key ="CSF_biased_gene_set").dropna(subset="names")
datCSFbiased.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_06_CD8_CSFnPB_biased_DEGs.csv")


# Score calculation

In [ ]:
### --- Get gene lists of CSF/PB-biased gene scores ---
CSFbiased_genescore = datCSFbiased[(datCSFbiased["logfoldchanges"]> 1) & (datCSFbiased["pvals_adj"]<0.01)]
PBbiased_genescore = datPBbiased[(datPBbiased["logfoldchanges"]> 1) & (datPBbiased["pvals_adj"]<0.01)]

# --- Export gene scores --- 
CSFbiased_genescore.to_csv("../../06_csv_outputs/DEG/CD8_CSF_biased_filtered_DEGs.csv")
PBbiased_genescore.to_csv("../../06_csv_outputs/DEG/CD8_PB_biased_filtered_DEGs.csv")

print(len(CSFbiased_genescore), " genes were CSF-biased")
print(len(PBbiased_genescore), " genes were CSF-biased")

# --- Calculate scores --- 
sc.tl.score_genes(sdata["gex"], 
                      CSFbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='CSF permissive score')

sc.tl.score_genes(sdata["gex"], 
                      PBbiased_genescore["names"].drop_duplicates().tolist(),  
                      score_name='PB resident score')
# --- Display scores --- 
p1 = sc.pl.umap(
    sdata["gex"], 
    color=["CSF permissive score", "PB resident score"], 
    size=10, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
p1.savefig("../../05_plots/13_final_plots/35_CD8_CSF_PB_biased_score_umap.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()

# Display CSF/PB-biased gene set

In [ ]:
print(f"{len(CSFbiased_genescore)} genes make up my CSF permissive score")
print(f"{len(PBbiased_genescore)} genes make up my PB resident score")

In [ ]:
sc.pl.dotplot(sdata_pb_filtered, CSFbiased_genescore["names"], groupby='top_vs_bot_clones', standard_scale="var", dendrogram=True, cmap="RdBu_r")

In [ ]:
sc.pl.dotplot(sdata_pb_filtered, PBbiased_genescore["names"], groupby='top_vs_bot_clones', dendrogram=True,standard_scale="var", cmap="RdBu_r")

# Correlation

In [ ]:
# Step 1: Extract relevant data
df = sdata["gex"].obs[["donor","clone_handle", "total_clone_count", "CSF permissive score", "PB resident score", "Compartment"]].copy()
df = df[df["Compartment"] == "PB"]
df = df[~df["donor"].isin(["Pt4", "Pt5"])]
df = df[df["total_clone_count"] > 12]
df["clone_handle"]= df["clone_handle"].cat.remove_unused_categories()

# Step 2: Compute average score per clone
clone_avg = df.groupby("clone_handle")[["PB resident score", "CSF permissive score"]].mean().reset_index()

# Step 3: Reorder based on clone_order
# clone_reordered = clone_avg.loc[clone_order].reset_index(drop=True)

# Step 4: Drop NaNs in CSF permissive score
# clone_reordered = clone_reordered.dropna(subset=["CSF permissive score"])

## Add information on CSF percentage
Merge_pct = pivot_table_comp.reset_index()
clone_avg = clone_avg.merge(Merge_pct, how="left", on="clone_handle")

## Filter on clones with some CSF-pct
# clone_avg = clone_avg[clone_avg["CSF.pct"] > 0]

# Prepare data
x_vals = clone_avg["CSF.pct"].values/100          # Before: np.arange(len(clone_reordered))
y_vals = clone_avg["CSF permissive score"].values   #clone_reordered["CSF permissive score"].values

# Compute Spearman correlation
rho, pval = spearmanr(x_vals, y_vals)

# Step 5: Plot using seaborn for regression with confidence interval
fig, ax = plt.subplots(figsize=(4.5, 4))

# Scatterplot + linear regression with CI
sb.regplot(x=x_vals, y=y_vals, ax=ax, ci=95, robust=True, scatter_kws={"color": "grey", "s": 40}, line_kws={"color": "black", "linestyle": "--"})

# Horizontal dashed line at y = 0
ax.axhline(y=0, linestyle='--', linewidth=0.5, color='lightgrey')

# Add Spearman correlation text
ax.text(0.01, 0.95,
        f"Spearman ρ = {rho:.2f}\np = {pval:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Formatting
ax.set_title("CSF permissive score vs clone rank\nwith Linear Regression + 95% CI and Spearman Correlation")
ax.set_xlabel("Fraction of cells in CSF/clone")
ax.set_ylabel("Average CSF-biased score per clone")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.set_ylim(0.75, 2)
ax.grid(False)

# Save the figure
fig.savefig("../../05_plots/13_final_plots/22_clone_score_regression_CI_spearman.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

In [ ]:
# Step 1: Extract relevant data
df = sdata["gex"].obs[["donor","clone_handle", "total_clone_count", "CSF permissive score", "PB resident score", "Compartment"]].copy()
df = df[df["Compartment"] == "PB"]
df = df[~df["donor"].isin(["Pt4", "Pt5"])]
df = df[df["total_clone_count"] > 12]
df["clone_handle"]= df["clone_handle"].cat.remove_unused_categories()

# Step 2: Compute average score per clone
clone_avg = df.groupby("clone_handle")[["PB resident score", "CSF permissive score"]].mean().reset_index()

# Step 3: Reorder based on clone_order
# clone_reordered = clone_avg.loc[clone_order].reset_index(drop=True)

# Step 4: Drop NaNs in CSF permissive score
# clone_reordered = clone_reordered.dropna(subset=["CSF permissive score"])

## Add information on CSF percentage
Merge_pct = pivot_table_comp.reset_index()
clone_avg = clone_avg.merge(Merge_pct, how="left", on="clone_handle")

## Filter on clones with some CSF-pct
# clone_avg = clone_avg[clone_avg["PB.pct"] > 0]

# Prepare data
x_vals = clone_avg["PB.pct"].values/100          # Before: np.arange(len(clone_reordered))
y_vals = clone_avg["PB resident score"].values   #clone_reordered["CSF permissive score"].values

# Compute Spearman correlation
rho, pval = spearmanr(x_vals, y_vals)

# Step 5: Plot using seaborn for regression with confidence interval
fig, ax = plt.subplots(figsize=(4.5, 4))

# Scatterplot + linear regression with CI
sb.regplot(x=x_vals, y=y_vals, ax=ax, ci=95, robust=True, scatter_kws={"color": "grey", "s": 40}, line_kws={"color": "black", "linestyle": "--"})

# Horizontal dashed line at y = 0
ax.axhline(y=0, linestyle='--', linewidth=0.5, color='lightgrey')

# Add Spearman correlation text
ax.text(0.01, 0.95,
        f"Spearman ρ = {rho:.2f}\np = {pval:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Formatting
ax.set_xlabel("Fraction of cells in PB")
ax.set_ylabel("Average PB-biased score")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
# ax.set_ylim(0.75, 2)
ax.grid(False)

# Save the figure
fig.savefig("../../05_plots/13_final_plots/22_clone_score_regression_CI_spearman_PB.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

# Outlier cells

In [ ]:
## 2 outlier clones
clone_avg[(clone_avg['CSF.pct'] > 60) & (clone_avg['CSF permissive score'] < 0.1) ]

In [ ]:
# Define clones to highlight
clones_of_interest = ["Pt17-26443", "Pt12-18654"]

# Define a new columns
sdata["gex"].obs["clone_highlight"] = pd.Categorical(np.where(sdata["gex"].obs["clone_handle"].isin(clones_of_interest), sdata["gex"].obs["clone_handle"], "unselected"))

# Palette
palette_highlight = {
    "unselected": "lightgrey",
    "Pt17-26443": "red",
    "Pt12-18654": "blue",
}

# Plot
sc.pl.umap(
    sdata["gex"],
    color="clone_highlight",
    groups=["Pt17-26443", "Pt12-18654"],
    palette=palette_highlight,
    size=50
)

In [ ]:
sdata["gex"].obs[sdata["gex"].obs["clone_handle"].isin(clones_of_interest)][["donor", "catpat"]].value_counts()

### CSF permissve score across clusters

In [ ]:
# Step 1: Extract relevant data
df = sdata["gex"].obs[["clone_handle", "CSF permissive score", "PB resident score", "Compartment", "L3_ann"]].copy()
# df = df[df["Compartment"] == "PB"]
# df = df[df["clone_handle"].isin(final_sets)]

# Compute mean score per L3_ann
l3_order = (
    df.groupby("L3_ann")["CSF permissive score"]
      .mean()
      .sort_values(ascending=False)
      .index
      .tolist()
)

fig, ax = plt.subplots(figsize=(6, 6))

# 1️⃣ Draw dots first
strip = sb.stripplot(
    data=df,
    x="L3_ann",
    y="CSF permissive score",
    order=l3_order,
    jitter=0.25,
    size=4,
    color="grey",
    alpha=0.5,
    zorder=1,
    ax=ax
)

# 👉 Rasterize the stripplot artists
for artist in strip.collections:
    artist.set_rasterized(True)

# 2️⃣ Draw boxplot on top (keep vector)
sb.boxplot(
    data=df,
    x="L3_ann",
    y="CSF permissive score",
    order=l3_order,
    showfliers=False,
    width=0.6,
    boxprops=dict(facecolor="white", edgecolor="black", linewidth=1.2),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.2),
    capprops=dict(color="black", linewidth=1.2),
    zorder=2,
    ax=ax
)

ax.set_xlabel("L3 annotation")
ax.set_ylabel("CSF permissive score")
ax.tick_params(axis="x", rotation=90)

fig.tight_layout()

fig.savefig("../../05_plots/13_final_plots/23_comparing_CSFbiased_score_across_clusters.pdf",
    bbox_inches="tight",
    format="pdf",
    dpi=400
)

plt.show()


## **CSF vs PB - hyperexpanded & shared - DEG per compartment**

In [ ]:
# ## Quick annotation of PB_shared_CSF
# ## We have two data frames with all CSF and PB clonotypes
# pb_set = set(sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "PB"]["clone_handle"])
# csf_set = set(sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "CSF"]["clone_handle"])

# #Define the overlap between these sets
# cOverlap = pb_set & csf_set

# sdata["gex"].obs["PB_shared_CSF"] = pd.Categorical(
#     np.where(sdata["gex"].obs["clone_handle"].isin(cOverlap), "shared", 
#             np.where(sdata["gex"].obs["clone_handle"].isin(pb_set), "PB",  
#                      np.where(sdata["gex"].obs["clone_handle"].isin(csf_set), "CSF", "Error"))))

In [ ]:
## Build the sdata object for DEG
sdata_deg = sdata.copy()

# Only work with shared clones that are hyper expanded
# mu.pp.filter_obs(sdata_deg["gex"], "PB_shared_CSF", lambda x: (x == "shared"))
mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
mu.pp.filter_obs(sdata_deg["gex"], "total_clone_count", lambda x: x > 12)

# Remove all unused categories
for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
    sdata_deg["gex"].obs[col] = sdata_deg["gex"].obs[col].cat.remove_unused_categories()

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# --- Filter genes before differential gene expression ---
## Define the filtered to alter: 
sdata_deg_filtered = sdata_deg["gex"].copy()

## --- Expression in CSF T cells
adata = sdata_deg["gex"][sdata_deg["gex"].obs["Compartment"] == "CSF"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

## --- Expression in PB-biased T cells
adata = sdata_deg["gex"][sdata_deg["gex"].obs["Compartment"] == "PB"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

#The total set of genes
gene_set_expressed = set(gene_list_10pct_1+gene_list_10pct_2)
print(len(gene_set_expressed))

# -----------------------------
# Step 2: exclude unwanted genes
# -----------------------------
genes = sdata_deg_filtered.var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))
exclude_genes = set(genes[exclude_mask])
genes_to_keep = sorted(gene_set_expressed - exclude_genes)
print("Final genes:", len(genes_to_keep))

# -----------------------------
# Step 3: subset AnnData correctly
# -----------------------------
sdata_deg_filtered = sdata_deg_filtered[:, genes_to_keep].copy()

# Perform naive differential gene expression
sc.tl.rank_genes_groups(sdata_deg_filtered, groupby="Compartment", groups=["PB"], reference="CSF", key_added="PB_gset")
sc.tl.rank_genes_groups(sdata_deg_filtered, groupby="Compartment", groups=["CSF"], reference="PB", key_added="CSF_gset")

#Visualize:
sc.pl.rank_genes_groups_dotplot(
      sdata_deg_filtered, groupby="Compartment", standard_scale="var", n_genes=15, key="PB_gset")
sc.pl.rank_genes_groups_dotplot(
      sdata_deg_filtered, groupby="Compartment", standard_scale="var", n_genes=15, key="CSF_gset")

# Export CSF and PB-specific gene sets

In [ ]:
## Get two dataframes of positive lfc values associated with CSF-biased and PB-biased T cells. Concatenate them and plot
datCSF = sc.get.rank_genes_groups_df(sdata_deg_filtered, group = None, key ="CSF_gset").dropna(subset="names")
datCSF.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_06_CD8_CSF_gset.csv")

datPB = sc.get.rank_genes_groups_df(sdata_deg_filtered, group = None,key ="PB_gset").dropna(subset="names")
datPB.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_06_CD8_PB_gset.csv")



### XY plot (CSF vs PB   &   CSF-biased to PB-biased in PB)

In [ ]:
datPBbiased.sort_values("logfoldchanges")

In [ ]:
datCSFbiased.head(2)

In [ ]:
datCSF.head(2)

In [ ]:
datPB.head(2)

In [ ]:
## Preprocess dataframes
PB_bias_up = datPBbiased[datPBbiased["logfoldchanges"] > 0]
CSF_bias_up = datCSFbiased[datCSFbiased["logfoldchanges"] > 0]
PB_up = datPB[datPB["logfoldchanges"] > 0]
CSF_up = datCSF[datCSF["logfoldchanges"] > 0]

## Transform the PB log2fc values to negative values
PB_up["logfoldchanges"] =-1*PB_up["logfoldchanges"] 
PB_bias_up["logfoldchanges"] =-1*PB_bias_up["logfoldchanges"] 

## Concatenate dataframes from the same analyses
Compartment_up = pd.concat([CSF_up, PB_up])
Bias_up = pd.concat([CSF_bias_up, PB_bias_up])

## Select and rename columns
Compartment_up = Compartment_up[["names", "logfoldchanges", "pvals_adj"]].rename(
    columns={
        "names": "gene",
        "logfoldchanges": "Compartment_LogFC",
        "pvals_adj": "Compartment_p_adj"
    }
)

Bias_up = Bias_up[["names", "logfoldchanges", "pvals_adj"]].rename(
    columns={
        "names": "gene",
        "logfoldchanges": "Biased_LogFC",
        "pvals_adj": "Biased_p_adj"
    }
)

# Merge
plot_df = Compartment_up.merge(Bias_up, how="outer", on="gene")

# Fill NaNs specifically
plot_df["Compartment_LogFC"] = plot_df["Compartment_LogFC"].fillna(0)
plot_df["Biased_LogFC"] = plot_df["Biased_LogFC"].fillna(0)
plot_df["Compartment_p_adj"] = plot_df["Compartment_p_adj"].fillna(1)
plot_df["Biased_p_adj"] = plot_df["Biased_p_adj"].fillna(1)



In [ ]:
# Define significance threshold
alpha = 0.05

# Assign colors based on significance
def get_color(row):
    sig_specific = row["Compartment_p_adj"] < alpha
    sig_biased = row["Biased_p_adj"] < alpha
    if sig_specific and sig_biased:
        return "red"
    elif sig_specific:
        return "blue"
    elif sig_biased:
        return "orange"
    else:
        return "gray"

plot_df["color"] = plot_df.apply(get_color, axis=1)

# Remove genes that are not significant in any
plot_df = plot_df[plot_df["color"] != "gray"]
plot_df.to_csv("../../06_csv_outputs/DEG/2026_6_CSFvPB_and_CSFbiased_vs_PBbiased_fulltable_non_signif_excluded.csv")

from adjustText import adjust_text

# Plot
plt.figure(figsize=(12, 9))
plt.scatter(
    plot_df["Compartment_LogFC"],
    plot_df["Biased_LogFC"],
    c=plot_df["color"],
    s=80,
    edgecolor="black"
)

# Add gene labels for strong log fold changes 
texts = []
highlight_genes1 = plot_df[(plot_df["Compartment_p_adj"] < alpha) & (plot_df["Biased_p_adj"] < alpha)].nlargest(10, "Compartment_LogFC")["gene"]
highlight_genes2 = plot_df[(plot_df["Compartment_p_adj"] < alpha) & (plot_df["Biased_p_adj"] < alpha)].nsmallest(10, "Compartment_LogFC")["gene"]
highlight_genes3 = plot_df[(plot_df["Compartment_p_adj"] < alpha)].nlargest(10, "Compartment_LogFC")["gene"]
highlight_genes4 = plot_df[(plot_df["Compartment_p_adj"] < alpha)].nsmallest(10, "Compartment_LogFC")["gene"]
highlight_genes5 = plot_df[(plot_df["Biased_p_adj"] < alpha)].nlargest(10, "Biased_LogFC")["gene"]
highlight_genes6 = plot_df[(plot_df["Biased_p_adj"] < alpha)].nsmallest(10, "Biased_LogFC")["gene"]
highlight_genes = set(highlight_genes1).union(highlight_genes2).union(highlight_genes3).union(highlight_genes4).union(highlight_genes5).union(highlight_genes6)

# To annotate genes: 
for _, row in plot_df.iterrows():
    if row["gene"] in highlight_genes:
        texts.append(
            plt.text(
                row["Compartment_LogFC"],
                row["Biased_LogFC"],
                row["gene"],
                fontsize=9
            )
        )

# Add reference lines and labels
# plt.ylim(-10,10)
plt.axhline(0, color='grey', linestyle='--', linewidth=0.5)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.5)
plt.xlabel("<-- PB  vs    CSF -->\nExpression among CD8 T cells of different tissues")
plt.ylabel("<-- PB-biased   vs    CSF-biased -->\nExpression within PB")
plt.title("Among hyperexpanded cells (n > 20)\nNon-TCR, non-ribosomal genes\noutlier genes excluded")

# Legend outside to the right
legend_patches = [
    mpatches.Patch(color="red", label="p_adj < 0.05 in both"),
    mpatches.Patch(color="blue", label="p_adj < 0.05 between tissues"),
    mpatches.Patch(color="orange", label="p_adj < 0.05 in PB subsets\nCSF-biased vs PB-biased"),
]
plt.legend(
    handles=legend_patches,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5)
)

# # Adjust text positions to avoid overlap
adjust_text(texts, arrowprops=dict(arrowstyle="->", color='gray', lw=0.25))

plt.tight_layout()

# Save plot
plt.savefig(
    "../../05_plots/13_final_plots/25_XY_CSFvsPB_DEGs.pdf",
    bbox_inches="tight", format="pdf", dpi=400
)

plt.show()

## Inspect

In [ ]:
plot_df[plot_df["Compartment_LogFC"] < -2].sort_values("Biased_LogFC", ascending=False).head(2)

# Curated signatures
Proposal: 
- Curated GZMK+ memory / CSF-biased (GZMK, CD27, GPR183, CD28, PASK, CCR7)
- Curated GZMB+ memory / PB-biased (LINC02384, GZMB, GNLY, TYROBP, ZNF683) 

In [ ]:
sc.pl.dotplot(sdata_pb_filtered, ["GZMK", "CD27", "GPR183", "CD28", "PASK", "CCR7", "LINC02384", "GZMB", "GNLY", "TYROBP", "ZNF683", "CX3CR1"], groupby='top_vs_bot_clones', standard_scale="var", dendrogram=True, cmap="RdBu_r")

In [ ]:
# Plot
sc.pl.umap(
    sdata["gex"],
    color = ["GZMK", "CD27", "GPR183", "CD28", "PASK", "CCR7"],
    size=40,
    cmap="RdBu_r"
)

# Derive a useful signature for Jasim

In [ ]:
## Try log2fc > 1 and p_adj < 0.01
List_CSFbiased_TEM = plot_df[(plot_df["Compartment_LogFC"] > 1) & (plot_df["Biased_LogFC"] > 1) & (plot_df["Compartment_p_adj"] < 0.01) & (plot_df["Biased_p_adj"] < 0.01)]
List_CSFbiased_TEM.to_csv("../../06_csv_outputs/DEG/2026_6_Signature_CSFbiased_TEMs_ForSpatial.csv")
List_CSFbiased_TEM

In [ ]:
## Try log2fc > 1 and p_adj < 0.01
List_PBbiased_TE = plot_df[(plot_df["Compartment_LogFC"] < -1) & (plot_df["Biased_LogFC"] < -1) & (plot_df["Compartment_p_adj"] < 0.01) & (plot_df["Biased_p_adj"] < 0.01)]
List_PBbiased_TE.to_csv("../../06_csv_outputs/DEG/2026_6_Signature_PBbiased_TEs_ForSpatial.csv")
List_PBbiased_TE.head(2)

# Expression in CSF (MS vs HD)

In [ ]:
## Build the sdata object for DEG
sdata_deg = sdata.copy()

# Only work with shared clones that are hyper expanded
# mu.pp.filter_obs(sdata_deg["gex"], "PB_shared_CSF", lambda x: (x == "shared"))
mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
mu.pp.filter_obs(sdata_deg["gex"], "total_clone_count", lambda x: x > 12)
mu.pp.filter_obs(sdata_deg["gex"], "Compartment", lambda x: x == "CSF")

# Remove all unused categories
for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
    sdata_deg["gex"].obs[col] = sdata_deg["gex"].obs[col].cat.remove_unused_categories()

In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Differential gene expression
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

# --- Filter genes before differential gene expression ---
## Define the filtered to alter: 
sdata_deg_filtered = sdata_deg["gex"].copy()

## --- Expression in CSF T cells
adata = sdata_deg["gex"][sdata_deg["gex"].obs["catpat"] == "MS"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_1 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

## --- Expression in PB-biased T cells
adata = sdata_deg["gex"][sdata_deg["gex"].obs["catpat"] == "NIC"].copy()
X = adata.X

# Compute fraction_pos and mean_counts
if not isinstance(X, np.ndarray):
    fraction_pos = np.array((X > 0).sum(axis=0)).flatten() / X.shape[0]
    mean_counts = np.array(X.mean(axis=0)).flatten()
else:
    fraction_pos = (X > 0).sum(axis=0) / X.shape[0]
    mean_counts = X.mean(axis=0)

fraction_df = pd.DataFrame({
    "names": adata.var_names,
    "fraction_pos": fraction_pos,
    "mean_counts": mean_counts
})

gene_list_10pct_2 = fraction_df[fraction_df["fraction_pos"] > 0.15]["names"].tolist() 

#The total set of genes
gene_set_expressed = set(gene_list_10pct_1+gene_list_10pct_2)
print(len(gene_set_expressed))

# -----------------------------
# Step 2: exclude unwanted genes
# -----------------------------
genes = sdata_deg_filtered.var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))
exclude_genes = set(genes[exclude_mask])
genes_to_keep = sorted(gene_set_expressed - exclude_genes)
print("Final genes:", len(genes_to_keep))

# -----------------------------
# Step 3: subset AnnData correctly
# -----------------------------
sdata_deg_filtered = sdata_deg_filtered[:, genes_to_keep].copy()

# Perform naive differential gene expression
sc.tl.rank_genes_groups(sdata_deg_filtered, groupby="catpat", groups=["MS"], reference="NIC", key_added="InCSF_MS_gset")
sc.tl.rank_genes_groups(sdata_deg_filtered, groupby="catpat", groups=["NIC"], reference="MS", key_added="InCSF_NIC_gset")

#Visualize:
sc.pl.rank_genes_groups_dotplot(
      sdata_deg_filtered, groupby="catpat", standard_scale="var", n_genes=15, key="InCSF_MS_gset")
sc.pl.rank_genes_groups_dotplot(
      sdata_deg_filtered, groupby="catpat", standard_scale="var", n_genes=15, key="InCSF_NIC_gset")

## Get two dataframes of positive lfc values associated with CSF-biased and PB-biased T cells. Concatenate them and plot
datMS_InCSF = sc.get.rank_genes_groups_df(sdata_deg_filtered, group = None, key ="InCSF_MS_gset").dropna(subset="names")
datMS_InCSF.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_06_CD8_CSF_InCSF_MS_gset.csv")

datNIC_InCSF = sc.get.rank_genes_groups_df(sdata_deg_filtered, group = None,key ="InCSF_NIC_gset").dropna(subset="names")
datNIC_InCSF.to_csv("..\\..\\06_csv_outputs\\DEG\\2026_06_CD8_InCSF_NIC.csv")



In [ ]:
datNIC_InCSF.head(5)

In [ ]:
datMS_InCSF.head(10)

In [ ]:
fig = sc.pl.violin(sdata_deg_filtered, keys=["AC004448.2", "IGLV2-14", "HLA-DRA", "CD74"], size=2, groupby='donor', rotation=90)

# CSF-biased score calculated via deseq2 (pseudobulked per donor)

### Pseudobulked using Decoupler's version of Deseq2
https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html

The pseudo-bulk approach involves the following steps:

    Subsetting the cell type of interest

    Extracting their raw integer counts

    Summing their counts per gene into a single profile if they pass quality NIC

Then, DEA can be performed if there are at least two biological replicates per condition (more replicates are recommended).

Pseudobulking can easily be performed using the function decoupler.pp.pseudobulk(). In this example, the counts are just summed, though other modes such as the mean or any custom aggregation function are available. For more information, refer to the mode argument.

In [ ]:
sdata["gex"].obs["top_vs_bot_clones"].value_counts()

In [ ]:
## ---- Define the data ---- 
sdata_bulk = sdata.copy()

mu.pp.filter_obs(sdata_bulk["gex"], "Compartment", lambda x: (x == "PB")) 
mu.pp.filter_obs(sdata_bulk["gex"], "top_vs_bot_clones", lambda x: (x != "unselected")) 

# Remove all unused categories
for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
    sdata_deg["gex"].obs[col] = sdata_deg["gex"].obs[col].cat.remove_unused_categories()

# Create a mask for genes to exclude
genes = sdata_bulk["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_bulk = sdata_bulk["gex"][:, genes_to_keep].copy()

## Bring the raw counts into .X
sdata_bulk.X = sdata_bulk.layers["counts"].copy()

In [ ]:
## ---- Use decoupler to pseudobulk cells ----
pdata = dc.pp.pseudobulk(
    adata=sdata_bulk,
    sample_col="donor",
    groups_col="top_vs_bot_clones",
    mode="sum",
)

pdata

In [ ]:
pdata.obs

In [ ]:
dc.pp.filter_samples(pdata, min_cells=5, min_counts=1000)

In [ ]:
pdata.obs

In [ ]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=pdata,
    design_factors=["top_vs_bot_clones"],
    refit_cooks=True,
    inference=inference,
)

# Compute LFCs
dds.deseq2()

# Extract contrast between conditions
stat_res = DeseqStats(dds, contrast=["top_vs_bot_clones", "PB-biased", "CSF-biased"], inference=inference)

# Compute Wald test
stat_res.summary()

In [ ]:
# Extract results
results_df = stat_res.results_df
results_df[results_df["padj"] < 0.05].sort_values("log2FoldChange", ascending=True).head(50)

In [ ]:
results_df[results_df.index == "CXCR4"]

In [ ]:
dc.pl.volcano(results_df, x="log2FoldChange", y="pvalue", top=20, figsize =(6,6))
plt.show()

In [ ]:
### --- Get gene lists of CSF/PB-biased gene scores ---
list_CSFbias = results_df[(results_df["log2FoldChange"]< -1) & (results_df["padj"]< 0.05)].index.tolist()
list_PBbias = results_df[(results_df["log2FoldChange"]> 1) & (results_df["padj"]< 0.05)].index.tolist()

#Remove sex-specific genes

sc.tl.score_genes(sdata["gex"], 
                      list_CSFbias,  
                      score_name='CSF permissive score (DESeq2)')

sc.tl.score_genes(sdata["gex"], 
                      list_PBbias,  
                      score_name='PB resident score (DESeq2)')

#Export genes making up each score
# CSF_permissive_score.to_csv("../../06_csv_outputs/DEG/CSF_permisiveness_score.csv")
# PB_resident_score.to_csv("../../06_csv_outputs/DEG/PB_resident_score.csv")

In [ ]:
print(f"{len(list_CSFbias)} genes make up my CSF permissive score (DESeq2)")
print(f"{len(list_PBbias)} genes make up my PB resident score (DESeq2)")

In [ ]:
p1 = sc.pl.umap(
    sdata["gex"], 
    color=["CSF permissive score", "CSF permissive score (DESeq2)", "PB resident score", "PB resident score (DESeq2)"], 
    size=10, 
    add_outline=False,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0,
    show=False,
    return_fig=True,
)
plt.show()
p1.savefig("../../05_plots/13_final_plots/18_gene_scores_CSFbiased_and_PBbiased.pdf", bbox_inches='tight', format="pdf", dpi = 400)

# CSF biased score per clone and per donor

### Define a list of  clones present in both CSF and PB

In [ ]:
listCSF = sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "CSF"]["clone_handle"].drop_duplicates().tolist()
listPB = sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "PB"]["clone_handle"].drop_duplicates().tolist()

# Clones present in both compartments
overlap = list(set(listCSF) & set(listPB))

In [ ]:
## Build the sdata object for DEG
sdata_deg = sdata.copy()

# Only work with shared clones that are hyper expanded
# mu.pp.filter_obs(sdata_deg["gex"], "clone_handle", lambda x: (x.isin(overlap)))
mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
mu.pp.filter_obs(sdata_deg["gex"], "total_clone_count", lambda x: x > 12)

# Remove all unused categories
for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
    sdata_deg["gex"].obs[col] = sdata_deg["gex"].obs[col].cat.remove_unused_categories()

# Step 1: Create a new dataframe with relevant columns
avg_df = sdata_deg["gex"].obs[["clone_handle", "donor", "Compartment", "CSF permissive score", "catpat"]]

# Step 2: Group and calculate average expression per group
avg_df = avg_df.groupby(["clone_handle", "donor", "catpat", "Compartment"])["CSF permissive score"].mean().reset_index().drop_duplicates().dropna(subset="CSF permissive score")

# Count unique clones
num_clones = avg_df["clone_handle"].nunique()

# Prepare paired data for Wilcoxon test
# Pivot avg_df to have columns PB and CSF expression per clone_handle
pivot_df = avg_df.pivot(index="clone_handle", columns="Compartment", values="CSF permissive score").dropna()

# Prepare data for stats and plotting
pivot_df = pivot_df.dropna()  # ensure no missing data

# Wilcoxon signed-rank test (paired)
stat, p_value = wilcoxon(pivot_df["PB"], pivot_df["CSF"])

# Format p-value text
if p_value < 0.001:
    p_text = "p < 0.001"
else:
    p_text = f"p = {p_value:.3f}"

# Melt for seaborn
df_long = pivot_df.reset_index().melt(id_vars="clone_handle", value_vars=["CSF", "PB"],
                               var_name="Compartment", value_name="Value")

plt.figure(figsize=(2.5,4))

# Boxplot with custom colors
sb.boxplot(data=df_long, x="Compartment", y="Value", 
           palette={"CSF": "#4B8AC1", "PB": "#C06A6A"}, 
           showfliers=False)

# Dots for individual points
sb.stripplot(data=df_long, x="Compartment", y="Value", color="black", size=6, jitter=False)

# Connect paired points with lines
for clone in df_long["clone_handle"].unique():
    paired_vals = df_long[df_long["clone_handle"] == clone].sort_values("Compartment")
    plt.plot([0, 1], paired_vals["Value"], color="gray", alpha=0.7, linewidth=1)

plt.title(f"CSF-biased score\n {p_text}")
plt.ylabel("Average expression")
plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/51_CSFbiased_comparison_CSF_v_PB.pdf", bbox_inches='tight', format="pdf", dpi = 300)
plt.show()


In [ ]:
# Number of unique clones
num_clones = avg_df["clone_handle"].nunique()

# Colors
compartment_colors = {"MS": "#4B8AC1", "NIC": "#C06A6A"}

# Compute Mann–Whitney U tests
stats = {}

for compartment, sub in avg_df.groupby("Compartment"):

    ms = sub.loc[sub["catpat"] == "MS", "CSF permissive score"]
    nic = sub.loc[sub["catpat"] == "NIC", "CSF permissive score"]

    n_ms = sub.loc[sub["catpat"] == "MS", "clone_handle"].nunique()
    n_nic = sub.loc[sub["catpat"] == "NIC", "clone_handle"].nunique()

    if len(ms) > 0 and len(nic) > 0:
        _, p = mannwhitneyu(ms, nic, alternative="two-sided")
    else:
        p = np.nan

    stats[compartment] = {"p": p, "n_ms": n_ms, "n_nic": n_nic}

# Create boxplots
g = sb.catplot(
    data=avg_df,
    x="catpat",
    y="CSF permissive score",
    col="Compartment",
    kind="box",
    palette=compartment_colors,
    showfliers=False,
    height=5,
    aspect=0.8,
)

# Overlay paired clone trajectories and points
for ax, compartment in zip(g.axes.flat, g.col_names):

    sub = avg_df[avg_df["Compartment"] == compartment]

    # Individual clone averages
    sb.stripplot(
        data=sub,
        x="catpat",
        y="CSF permissive score",
        color="black",
        jitter=0.15,
        size=5,
        alpha=0.7,
        ax=ax,
        zorder=3,
    )

    # Statistics
    p = stats[compartment]["p"]
    n_ms = stats[compartment]["n_ms"]
    n_nic = stats[compartment]["n_nic"]

    if np.isnan(p):
        p_text = "p = NA"
    elif p < 0.001:
        p_text = f"p = {p:.2e}"
    else:
        p_text = f"p = {p:.3f}"

    ax.set_title(
        f"{compartment}\n"
        f"{p_text}"
    )

g.set_axis_labels("Patient group", "Average expression")
#Set figure size
g.fig.set_size_inches(4.5, 5)
g.fig.suptitle(
    f"Among hyperexpanded clones (>12 cells)\n"
    f"{num_clones} unique shared clones present in both compartments",
    fontsize=14,
    y=1.03,
)

plt.tight_layout()
plt.savefig("..\\..\\05_plots\\08_AfterAntwerp\\CSFbiased_comparison_CSF_v_PB2.pdf", bbox_inches='tight', format="pdf", dpi = 300)
plt.show()

In [ ]:
# Aggregate
avg_df = (
    sdata_deg["gex"].obs[
        ["donor", "Compartment", "catpat", "CSF permissive score"]
    ]
    .groupby(["donor", "catpat", "Compartment"])["CSF permissive score"]
    .mean()
    .reset_index()
    .dropna()
)

num_clones = avg_df["donor"].nunique()

# Stats per compartment
stats = {}

for compartment, sub in avg_df.groupby("Compartment"):

    ms = sub.loc[sub["catpat"] == "MS", "CSF permissive score"]
    nic = sub.loc[sub["catpat"] == "NIC", "CSF permissive score"]

    n_ms = sub.loc[sub["catpat"] == "MS", "donor"].nunique()
    n_nic = sub.loc[sub["catpat"] == "NIC", "donor"].nunique()

    if len(ms) > 0 and len(nic) > 0:
        _, p = mannwhitneyu(ms, nic, alternative="two-sided")
    else:
        p = np.nan

    stats[compartment] = {"p": p, "n_ms": n_ms, "n_nic": n_nic}

# Plot
g = sb.catplot(
    data=avg_df,
    x="catpat",
    y="CSF permissive score",
    col="Compartment",
    kind="box",
    linewidth=0.5, 
    palette=compartment_colors,
    showfliers=False,
    height=5,
    aspect=0.8,
)

#Set figure size
g.fig.set_size_inches(4.25, 5)

for ax, compartment in zip(g.axes.flat, g.col_names):

    sub = avg_df[avg_df["Compartment"] == compartment]

    # Connect clones
    for _, clone in sub.groupby("donor"):
        if clone["catpat"].nunique() == 2:
            clone = clone.set_index("catpat").loc[["MS", "NIC"]]
            ax.plot(
                [0, 1],
                clone["CSF permissive score"],
                color="gray",
                alpha=0.6,
                lw=1,
                zorder=2,
            )

    sb.stripplot(
        data=sub,
        x="catpat",
        y="CSF permissive score",
        color="black",
        jitter=0.15,
        linewidth=0.5,
        size=5,
        alpha=0.7,
        ax=ax,
        zorder=3,
    )

    p = stats[compartment]["p"]
    n_ms = stats[compartment]["n_ms"]
    n_nic = stats[compartment]["n_nic"]

    if np.isnan(p):
        p_text = "p = NA"
    elif p < 0.001:
        p_text = f"p = {p:.2e}"
    else:
        p_text = f"p = {p:.3f}"

    ax.set_title(
        f"{compartment}\n"
        f"MS: {n_ms} | NIC: {n_nic}\n"
        f"{p_text}"
    )

g.set_axis_labels(" ",f"Avg. expr. / donor")

g.fig.suptitle(
    f"CSF-biased score, (n> 12 cells)\n"
    f"{num_clones} donors",
    y=1.03,
)

plt.tight_layout()
plt.savefig("..\\..\\05_plots\\08_AfterAntwerp\\CSFbiased_comparison_CSF_v_PB3.pdf", bbox_inches='tight', format="pdf", dpi = 300)
plt.show()

# Expression per clone

### Define a list of  clones present in both CSF and PB

In [ ]:
listCSF = sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "CSF"]["clone_handle"].drop_duplicates().tolist()
listPB = sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "PB"]["clone_handle"].drop_duplicates().tolist()

# Clones present in both compartments
overlap = list(set(listCSF) & set(listPB))

## GZMK per clone

In [ ]:
## Build the sdata object for DEG
sdata_deg = sdata.copy()

# Only work with shared clones that are hyper expanded
mu.pp.filter_obs(sdata_deg["gex"], "clone_handle", lambda x: (x.isin(overlap)))
mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
mu.pp.filter_obs(sdata_deg["gex"], "total_clone_count", lambda x: x > 12)


# Remove all unused categories
for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
    sdata_deg["gex"].obs[col] = sdata_deg["gex"].obs[col].cat.remove_unused_categories()

# Pull expression of GZMK into .obs
sdata_deg["gex"].obs["GZMK_expr"] = sdata_deg["gex"][:, "GZMK"].X.toarray().flatten()

# Step 1: Create a new dataframe with relevant columns
avg_df = sdata_deg["gex"].obs[["clone_handle", "donor", "Compartment", "GZMK_expr", "catpat"]]

# Step 3: Group and calculate average expression per group
avg_df = avg_df.groupby(["clone_handle", "donor", "catpat", "Compartment"])["GZMK_expr"].mean().reset_index().drop_duplicates().dropna(subset="GZMK_expr")

# Count unique clones
num_clones = avg_df["clone_handle"].nunique()

# Prepare paired data for Wilcoxon test
# Pivot avg_df to have columns PB and CSF expression per clone_handle
pivot_df = avg_df.pivot(index="clone_handle", columns="Compartment", values="GZMK_expr").dropna()

# Prepare data for stats and plotting
pivot_df = pivot_df.dropna()  # ensure no missing data

# Wilcoxon signed-rank test (paired)
stat, p_value = wilcoxon(pivot_df["PB"], pivot_df["CSF"])

# Format p-value text
if p_value < 0.001:
    p_text = "p < 0.001"
else:
    p_text = f"p = {p_value:.3f}"

# Melt for seaborn
df_long = pivot_df.reset_index().melt(id_vars="clone_handle", value_vars=["CSF", "PB"],
                               var_name="Compartment", value_name="Value")

plt.figure(figsize=(2.5,4))

# Boxplot with custom colors
sb.boxplot(data=df_long, x="Compartment", y="Value", 
           palette={"CSF": "#4B8AC1", "PB": "#C06A6A"}, 
           showfliers=False)

# Dots for individual points
sb.stripplot(data=df_long, x="Compartment", y="Value", color="black", size=6, jitter=False)

# Connect paired points with lines
for clone in df_long["clone_handle"].unique():
    paired_vals = df_long[df_long["clone_handle"] == clone].sort_values("Compartment")
    plt.plot([0, 1], paired_vals["Value"], color="gray", alpha=0.7, linewidth=1)

plt.title(f"Paired data across compartments\nWilcoxon signed-rank test {p_text}")
plt.ylabel("Average GZMK expression")
plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/51_GZMK_biased_comparison_CSF_v_PB.pdf", bbox_inches='tight', format="pdf", dpi = 300)
plt.show()


In [ ]:
# Number of unique clones
num_clones = avg_df["clone_handle"].nunique()

# Colors
compartment_colors = {"MS": "#4B8AC1", "NIC": "#C06A6A"}

# Compute Mann–Whitney U tests
stats = {}

for compartment, sub in avg_df.groupby("Compartment"):

    ms = sub.loc[sub["catpat"] == "MS", "GZMK_expr"]
    nic = sub.loc[sub["catpat"] == "NIC", "GZMK_expr"]

    n_ms = sub.loc[sub["catpat"] == "MS", "clone_handle"].nunique()
    n_nic = sub.loc[sub["catpat"] == "NIC", "clone_handle"].nunique()

    if len(ms) > 0 and len(nic) > 0:
        _, p = mannwhitneyu(ms, nic, alternative="two-sided")
    else:
        p = np.nan

    stats[compartment] = {"p": p, "n_ms": n_ms, "n_nic": n_nic}

# Create boxplots
g = sb.catplot(
    data=avg_df,
    x="catpat",
    y="GZMK_expr",
    col="Compartment",
    kind="box",
    palette=compartment_colors,
    showfliers=False,
    height=5,
    aspect=0.8,
)

# Overlay paired clone trajectories and points
for ax, compartment in zip(g.axes.flat, g.col_names):

    sub = avg_df[avg_df["Compartment"] == compartment]

    # Connect observations from the same clone (visual aid only)
    for _, clone in sub.groupby("clone_handle"):
        if clone["catpat"].nunique() == 2:
            clone = clone.set_index("catpat").loc[["MS", "NIC"]]
            ax.plot(
                [0, 1],
                clone["GZMK_expr"],
                color="gray",
                alpha=0.6,
                lw=1,
                zorder=2,
            )

    # Individual clone averages
    sb.stripplot(
        data=sub,
        x="catpat",
        y="GZMK_expr",
        color="black",
        jitter=0.15,
        size=5,
        alpha=0.7,
        ax=ax,
        zorder=3,
    )

    # Statistics
    p = stats[compartment]["p"]
    n_ms = stats[compartment]["n_ms"]
    n_nic = stats[compartment]["n_nic"]

    if np.isnan(p):
        p_text = "p = NA"
    elif p < 0.001:
        p_text = f"p = {p:.2e}"
    else:
        p_text = f"p = {p:.3f}"

    ax.set_title(
        f"{compartment}\n"
        f"MS: {n_ms} clones | NIC: {n_nic} clones\n"
        f"{p_text}"
    )

g.set_axis_labels("Patient group", "Average GZMK expression")
g.fig.suptitle(
    f"Among hyperexpanded clones (>12 cells)\n"
    f"{num_clones} unique shared clones present in both compartments",
    fontsize=14,
    y=1.03,
)

plt.tight_layout()
plt.savefig("..\\..\\05_plots\\08_AfterAntwerp\\GZMK_comparison_CSF_v_PB2.pdf", bbox_inches='tight', format="pdf", dpi = 300)
plt.show()

## Define function

In [ ]:
def plot_gene_csf_vs_pb(sdata, gene, overlap, save_path=None):
    """
    Compare average clone-level gene expression between CSF and PB
    using a paired Wilcoxon signed-rank test.

    Parameters
    ----------
    sdata : MuData
        MuData object containing the "gex" modality.
    gene : str
        Gene name (must exist in sdata["gex"].var_names).
    overlap : list
        List of clone_handle values shared between compartments.
    save_path : str or None, optional
        Path to save the figure. If None, the figure is not saved.
    """

    ## 
    listCSF = sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "CSF"]["clone_handle"].drop_duplicates().tolist()
    listPB = sdata["gex"].obs[sdata["gex"].obs["Compartment"] == "PB"]["clone_handle"].drop_duplicates().tolist()
    
    # Clones present in both compartments
    overlap = list(set(listCSF) & set(listPB))

    # Copy data
    sdata_deg = sdata.copy()

    # Only work with shared clones that are hyper expanded
    mu.pp.filter_obs(sdata_deg["gex"], "clone_handle", lambda x: (x.isin(overlap)))
    mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
    mu.pp.filter_obs(sdata_deg["gex"], "total_clone_count", lambda x: x > 12)
    
    # Only work with 5 clones per donor to have equal representation (top most expanded)
    # mu.pp.filter_obs(sdata_deg["gex"], "clone_handle", lambda x: (x.isin(top_clone_handles)))
    # mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))

    # Remove unused categories
    for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
        sdata_deg["gex"].obs[col] = (
            sdata_deg["gex"].obs[col]
            .cat.remove_unused_categories()
        )

    # Pull gene expression into obs
    expr_col = f"{gene}_expr"
    sdata_deg["gex"].obs[expr_col] = (
        sdata_deg["gex"][:, gene].X.toarray().flatten()
    )

    # Average expression per clone/donor/compartment
    avg_df = (
        sdata_deg["gex"].obs[
            ["clone_handle", "donor", "Compartment", expr_col]
        ]
        .groupby(["clone_handle", "donor", "Compartment"])[expr_col]
        .mean()
        .reset_index()
        .dropna()
    )

    num_clones = avg_df["clone_handle"].nunique()

    # Prepare paired data
    pivot_df = (
        avg_df.pivot(index="clone_handle",
                     columns="Compartment",
                     values=expr_col)
        .dropna()
    )

    # Wilcoxon signed-rank test
    _, p_value = wilcoxon(pivot_df["PB"], pivot_df["CSF"])

    if p_value < 0.001:
        p_text = f"p = {p_value:.2e}"
    else:
        p_text = f"p = {p_value:.3f}"

    # Long format for plotting
    df_long = (
        pivot_df.reset_index()
        .melt(
            id_vars="clone_handle",
            value_vars=["CSF", "PB"],
            var_name="Compartment",
            value_name="Value",
        )
    )

    # Plot
    plt.figure(figsize=(2.5, 4))

    sb.boxplot(
        data=df_long,
        x="Compartment",
        y="Value",
        linewidth=0.5, 
        palette={"CSF": "#4B8AC1", "PB": "#C06A6A"},
        showfliers=False,
    )

    sb.stripplot(
        data=df_long,
        x="Compartment",
        y="Value",
        color="black",
        linewidth=0.5, 
        size=6,
        jitter=False,
    )

    # Connect paired clones
    for clone in df_long["clone_handle"].unique():
        paired = (
            df_long[df_long["clone_handle"] == clone]
            .set_index("Compartment")
            .loc[["CSF", "PB"]]
        )

        plt.plot(
            [0, 1],
            paired["Value"],
            color="gray",
            alpha=0.7,
            linewidth=1,
        )

    plt.title(
        f"{gene}\n {p_text}")
    plt.ylabel(f"Avg. expr. / clone")
    plt.xlabel("")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="GZMK",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMK_comparison_CSF_v_PB.pdf",
)

## Define function (Compartment analysis with clones as replicates)

In [ ]:
def plot_gene_clone_deg(
    sdata,
    gene,
    overlap,
    min_clone_size=12,
    donors_exclude=("Pt4", "Pt5"),
    compartment_colors=None,
    save_path=None,
):
    """
    Clone-level DEG-style plot across compartments and patient groups.

    Parameters
    ----------
    sdata : MuData
    gene : str
        Gene to analyze.
    overlap : list
        Shared clone handles to include.
    min_clone_size : int
        Minimum total_clone_count threshold.
    donors_exclude : tuple
        Donors to exclude.
    compartment_colors : dict
        Color mapping for catpat groups.
    save_path : str or None
        Optional output path.
    """

    if compartment_colors is None:
        compartment_colors = {"MS": "#4B8AC1", "NIC": "#C06A6A"}

    expr_col = f"{gene}_expr"

    # Copy object
    sdata_deg = sdata.copy()

    # Filtering
    mu.pp.filter_obs(
        sdata_deg["gex"],
        "clone_handle",
        lambda x: x.isin(overlap),
    )

    mu.pp.filter_obs(
        sdata_deg["gex"],
        "donor",
        lambda x: ~x.isin(list(donors_exclude)),
    )

    mu.pp.filter_obs(
        sdata_deg["gex"],
        "total_clone_count",
        lambda x: x > min_clone_size,
    )
    
    # Only work with 5 clones per donor to have equal representation (top most expanded)
    # mu.pp.filter_obs(sdata_deg["gex"], "clone_handle", lambda x: (x.isin(top_clone_handles)))
    # mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))

    # Remove unused categories
    for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
        sdata_deg["gex"].obs[col] = (
            sdata_deg["gex"].obs[col].cat.remove_unused_categories()
        )

    # Expression
    sdata_deg["gex"].obs[expr_col] = (
        sdata_deg["gex"][:, gene].X.toarray().flatten()
    )

    # Aggregate
    avg_df = (
        sdata_deg["gex"].obs[
            ["clone_handle", "donor", "Compartment", "catpat", expr_col]
        ]
        .groupby(["clone_handle", "donor", "catpat", "Compartment"])[expr_col]
        .mean()
        .reset_index()
        .dropna()
    )

    num_clones = avg_df["clone_handle"].nunique()

    # Stats per compartment
    stats = {}

    for compartment, sub in avg_df.groupby("Compartment"):

        ms = sub.loc[sub["catpat"] == "MS", expr_col]
        nic = sub.loc[sub["catpat"] == "NIC", expr_col]

        n_ms = sub.loc[sub["catpat"] == "MS", "clone_handle"].nunique()
        n_nic = sub.loc[sub["catpat"] == "NIC", "clone_handle"].nunique()

        if len(ms) > 0 and len(nic) > 0:
            _, p = mannwhitneyu(ms, nic, alternative="two-sided")
        else:
            p = np.nan

        stats[compartment] = {"p": p, "n_ms": n_ms, "n_nic": n_nic}

    # Plot
    g = sb.catplot(
        data=avg_df,
        x="catpat",
        y=expr_col,
        col="Compartment",
        kind="box",
        linewidth=0.5, 
        palette=compartment_colors,
        showfliers=False,
        height=5,
        aspect=0.8,
    )

    #Set figure size
    g.fig.set_size_inches(4.5, 5)

    for ax, compartment in zip(g.axes.flat, g.col_names):

        sub = avg_df[avg_df["Compartment"] == compartment]

        # Connect clones
        for _, clone in sub.groupby("clone_handle"):
            if clone["catpat"].nunique() == 2:
                clone = clone.set_index("catpat").loc[["MS", "NIC"]]
                ax.plot(
                    [0, 1],
                    clone[expr_col],
                    color="gray",
                    alpha=0.6,
                    lw=1,
                    zorder=2,
                )

        sb.stripplot(
            data=sub,
            x="catpat",
            y=expr_col,
            color="black",
            jitter=0.15,
            linewidth=0.5,
            size=5,
            alpha=0.7,
            ax=ax,
            zorder=3,
        )

        p = stats[compartment]["p"]
        n_ms = stats[compartment]["n_ms"]
        n_nic = stats[compartment]["n_nic"]

        if np.isnan(p):
            p_text = "p = NA"
        elif p < 0.001:
            p_text = f"p = {p:.2e}"
        else:
            p_text = f"p = {p:.3f}"

        ax.set_title(
            f"{compartment}\n"
            f"MS: {n_ms} | NIC: {n_nic}\n"
            f"{p_text}"
        )

    g.set_axis_labels(" ",f"Avg. expr. / clone")

    g.fig.suptitle(
        f"Clone-level {gene} expression (> {min_clone_size} cells)\n"
        f"{num_clones} shared clones",
        y=1.03,
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.show()

    return avg_df, stats

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="GZMK",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMK_clone_DEG2.pdf",
)

## Define function (Compartment analysis with donors as replicates)

In [ ]:
def plot_gene_donor_deg(
    sdata,
    gene,
    overlap,
    min_clone_size=12,
    donors_exclude=("Pt4", "Pt5"),
    compartment_colors=None,
    save_path=None,
):
    """
    Clone-level DEG-style plot across compartments and patient groups.

    Parameters
    ----------
    sdata : MuData
    gene : str
        Gene to analyze.
    overlap : list
        Shared clone handles to include.
    min_clone_size : int
        Minimum total_clone_count threshold.
    donors_exclude : tuple
        Donors to exclude.
    compartment_colors : dict
        Color mapping for catpat groups.
    save_path : str or None
        Optional output path.
    """

    if compartment_colors is None:
        compartment_colors = {"MS": "#4B8AC1", "NIC": "#C06A6A"}

    expr_col = f"{gene}_expr"

    # Copy object
    sdata_deg = sdata.copy()

    # Filtering
    mu.pp.filter_obs(
        sdata_deg["gex"],
        "clone_handle",
        lambda x: x.isin(overlap),
    )

    mu.pp.filter_obs(
        sdata_deg["gex"],
        "donor",
        lambda x: ~x.isin(list(donors_exclude)),
    )

    mu.pp.filter_obs(
        sdata_deg["gex"],
        "total_clone_count",
        lambda x: x > min_clone_size,
    )
    
    # Only work with 5 clones per donor to have equal representation (top most expanded)
    # mu.pp.filter_obs(sdata_deg["gex"], "clone_handle", lambda x: (x.isin(top_clone_handles)))
    # mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))

    # Remove unused categories
    for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
        sdata_deg["gex"].obs[col] = (
            sdata_deg["gex"].obs[col].cat.remove_unused_categories()
        )

    # Expression
    sdata_deg["gex"].obs[expr_col] = (
        sdata_deg["gex"][:, gene].X.toarray().flatten()
    )

    # Aggregate
    avg_df = (
        sdata_deg["gex"].obs[
            ["donor", "Compartment", "catpat", expr_col]
        ]
        .groupby(["donor", "catpat", "Compartment"])[expr_col]
        .mean()
        .reset_index()
        .dropna()
    )

    num_clones = avg_df["donor"].nunique()

    # Stats per compartment
    stats = {}

    for compartment, sub in avg_df.groupby("Compartment"):

        ms = sub.loc[sub["catpat"] == "MS", expr_col]
        nic = sub.loc[sub["catpat"] == "NIC", expr_col]

        n_ms = sub.loc[sub["catpat"] == "MS", "donor"].nunique()
        n_nic = sub.loc[sub["catpat"] == "NIC", "donor"].nunique()

        if len(ms) > 0 and len(nic) > 0:
            _, p = mannwhitneyu(ms, nic, alternative="two-sided")
        else:
            p = np.nan

        stats[compartment] = {"p": p, "n_ms": n_ms, "n_nic": n_nic}

    # Plot
    g = sb.catplot(
        data=avg_df,
        x="catpat",
        y=expr_col,
        col="Compartment",
        kind="box",
        linewidth=0.5, 
        palette=compartment_colors,
        showfliers=False,
        height=5,
        aspect=0.8,
    )

    #Set figure size
    g.fig.set_size_inches(4.25, 5)

    for ax, compartment in zip(g.axes.flat, g.col_names):

        sub = avg_df[avg_df["Compartment"] == compartment]

        # Connect clones
        for _, clone in sub.groupby("donor"):
            if clone["catpat"].nunique() == 2:
                clone = clone.set_index("catpat").loc[["MS", "NIC"]]
                ax.plot(
                    [0, 1],
                    clone[expr_col],
                    color="gray",
                    alpha=0.6,
                    lw=1,
                    zorder=2,
                )

        sb.stripplot(
            data=sub,
            x="catpat",
            y=expr_col,
            color="black",
            jitter=0.15,
            linewidth=0.5,
            size=5,
            alpha=0.7,
            ax=ax,
            zorder=3,
        )

        p = stats[compartment]["p"]
        n_ms = stats[compartment]["n_ms"]
        n_nic = stats[compartment]["n_nic"]

        if np.isnan(p):
            p_text = "p = NA"
        elif p < 0.001:
            p_text = f"p = {p:.2e}"
        else:
            p_text = f"p = {p:.3f}"

        ax.set_title(
            f"{compartment}\n"
            f"MS: {n_ms} | NIC: {n_nic}\n"
            f"{p_text}"
        )

    g.set_axis_labels(" ",f"Avg. expr. / donor")

    g.fig.suptitle(
        f"Clone-level {gene} expression (> {min_clone_size} cells)\n"
        f"{num_clones} donors",
        y=1.03,
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches="tight", dpi=300)

    plt.show()

    return avg_df, stats

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="GZMK",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMK_clone_DEG3.pdf",
)

## Compare HLA-DRB1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="HLA-DRB1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/HLA-DRB1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="HLA-DRB1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/HLA-DRB1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="HLA-DRB1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/HLA-DRB1_clone_DEG3.pdf",
)

## Compare CXCR3

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CXCR3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CXCR3_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CXCR3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CXCR3_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CXCR3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CXCR3_clone_DEG3.pdf",
)

## Compare CXCR4

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CXCR4",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CXCR4_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CXCR4",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CXCR4_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CXCR4",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CXCR4_clone_DEG3.pdf",
)

## Compare GZMA

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="GZMA",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMA_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="GZMA",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMA_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="GZMA",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMA_clone_DEG3.pdf",
)

## Compare GPR183

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="GPR183",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GPR183_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="GPR183",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GPR183_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="GPR183",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GPR183_clone_DEG3.pdf",
)

## Compare CD27

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CD27",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD27_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CD27",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD27_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CD27",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD27_clone_DEG3.pdf",
)

## Compare CD28

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CD28",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD28_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CD28",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD28_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CD28",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD28_clone_DEG3.pdf",
)

## Compare "IGLV2-14"

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="IGLV2-14",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/IGLV2-14_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="IGLV2-14",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/IGLV2-14_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="IGLV2-14",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/IGLV2-14_clone_DEG3.pdf",
)

## Compare PASK

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="PASK",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PASK_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="PASK",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PASK_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="PASK",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PASK_clone_DEG3.pdf",
)

## Compare SELL

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="SELL",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SELL_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="SELL",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SELL_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="SELL",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SELL_clone_DEG3.pdf",
)

## Compare CCR7

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CCR7",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCR7_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CCR7",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCR7_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CCR7",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCR7_clone_DEG3.pdf",
)

## Compare COTL1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="COTL1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/COTL1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="COTL1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/COTL1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="COTL1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/COTL1_clone_DEG3.pdf",
)

## Compare DUSP2

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="DUSP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/DUSP2_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="DUSP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/DUSP2_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="DUSP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/DUSP2_clone_DEG3.pdf",
)

## Compare NELL2

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="NELL2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/NELL2_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="NELL2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/NELL2_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="NELL2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/NELL2_clone_DEG3.pdf",
)

## Compare CLDND1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CLDND1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CLDND1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CLDND1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CLDND1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CLDND1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CLDND1_clone_DEG3.pdf",
)

## Compare GZMH

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="GZMH",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMH_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="GZMH",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMH_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="GZMH",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMH_clone_DEG3.pdf",
)

## Compare GZMB

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="GZMB",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMB_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="GZMB",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMB_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="GZMB",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GZMB_clone_DEG3.pdf",
)

## Compare ID3

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="ID3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ID3_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="ID3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ID3_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="ID3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ID3_clone_DEG3.pdf",
)

## Compare PRF1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="PRF1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PRF1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="PRF1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PRF1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="PRF1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PRF1_clone_DEG3.pdf",
)

## Compare PDCD1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="PDCD1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PDCD1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="PDCD1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PDCD1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="PDCD1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PDCD1_clone_DEG3.pdf",
)

## Compare LAG3

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="LAG3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/LAG3_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="LAG3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/LAG3_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="LAG3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/LAG3_clone_DEG3.pdf",
)

## Compare GNLY

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="GNLY",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GNLY_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="GNLY",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GNLY_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="GNLY",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GNLY_clone_DEG3.pdf",
)

## Compare PCBP1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="PCBP1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PCBP1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="PCBP1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PCBP1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="PCBP1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PCBP1_clone_DEG3.pdf",
)

## Compare PPP1R14B

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="PPP1R14B",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PPP1R14B_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="PPP1R14B",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PPP1R14B_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="PPP1R14B",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PPP1R14B_clone_DEG3.pdf",
)

## Compare ITGA4

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="ITGA4",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ITGA4_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="ITGA4",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ITGA4_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="ITGA4",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ITGA4_clone_DEG3.pdf",
)

## Compare CCR5

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CCR5",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCR5_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CCR5",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCR5_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CCR5",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCR5_clone_DEG3.pdf",
)

# T cells with B cell transcripts
IGKV genes:
https://www.imgt.org/IMGTrepertoire/LocusGenes/genetable/autotable.php?group=IGKV&species=Homo+sapiens
IGLV genes:
https://www.imgt.org/IMGTrepertoire/LocusGenes/genetable/autotable.php?group=IGLV&species=Homo+sapiens

In [ ]:
# ----------------------------
# 1. Define IGK + IGL gene sets
# ----------------------------
igk_genes = [g for g in sdata["gex"].var_names if g.startswith("IGKV")]
igl_genes = [g for g in sdata["gex"].var_names if g.startswith("IGLV")]
igh_genes = [g for g in sdata["gex"].var_names if g.startswith("IGHV")]

bcr_genes = igl_genes 


# ----------------------------
# 2. Copy data + filter
# ----------------------------
sdata_deg = sdata.copy()

mu.pp.filter_obs(sdata_deg["gex"], "clone_handle", lambda x: x.isin(overlap))
mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: ~x.isin(["Pt4", "Pt5"]))
mu.pp.filter_obs(sdata_deg["gex"], "total_clone_count", lambda x: x > 12)

# Remove unused categories
for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
    sdata_deg["gex"].obs[col] = sdata_deg["gex"].obs[col].cat.remove_unused_categories()


# ----------------------------
# 3. Extract BCR gene expression
# ----------------------------
X = sdata_deg["gex"][:, bcr_genes].X

# handle sparse matrices safely
if not hasattr(X, "toarray"):
    X = np.array(X)
else:
    X = X.toarray()


# ----------------------------
# 4. Add per-gene expression into .obs
# ----------------------------
for i, gene in enumerate(bcr_genes):
    sdata_deg["gex"].obs[gene + "_expr"] = X[:, i]


# ----------------------------
# 5. Create combined BCR score
# ----------------------------
sdata_deg["gex"].obs["BCR_related_genes"] = X.sum(axis=1)

In [ ]:
# ----------------------------
# Clone-level aggregation
# ----------------------------
avg_df = (
    sdata_deg["gex"].obs[
        ["clone_handle", "donor", "Compartment", "catpat", "BCR_related_genes"]
    ]
    .groupby(["clone_handle", "donor", "catpat", "Compartment"])["BCR_related_genes"]
    .mean()
    .reset_index()
    .dropna()
)


# ----------------------------
# Stats
# ----------------------------
num_clones = avg_df["clone_handle"].nunique()

compartment_colors = {"MS": "#4B8AC1", "NIC": "#C06A6A"}

stats = {}

for compartment, sub in avg_df.groupby("Compartment"):

    ms = sub.loc[sub["catpat"] == "MS", "BCR_related_genes"]
    nic = sub.loc[sub["catpat"] == "NIC", "BCR_related_genes"]

    n_ms = sub.loc[sub["catpat"] == "MS", "clone_handle"].nunique()
    n_nic = sub.loc[sub["catpat"] == "NIC", "clone_handle"].nunique()

    if len(ms) > 0 and len(nic) > 0:
        _, p = mannwhitneyu(ms, nic, alternative="two-sided")
    else:
        p = np.nan

    stats[compartment] = {"p": p, "n_ms": n_ms, "n_nic": n_nic}


# ----------------------------
# Plot
# ----------------------------
g = sb.catplot(
    data=avg_df,
    x="catpat",
    y="BCR_related_genes",
    col="Compartment",
    kind="box",
    palette=compartment_colors,
    showfliers=False,
    height=5,
    aspect=0.8,
)

# ----------------------------
# Overlay clone trajectories
# ----------------------------
for ax, compartment in zip(g.axes.flat, g.col_names):

    sub = avg_df[avg_df["Compartment"] == compartment]

    sb.stripplot(
        data=sub,
        x="catpat",
        y="BCR_related_genes",
        color="black",
        jitter=0.15,
        size=5,
        alpha=0.7,
        ax=ax,
        zorder=3,
    )

    # ----------------------------
    # Stats annotation
    # ----------------------------
    p = stats[compartment]["p"]
    n_ms = stats[compartment]["n_ms"]
    n_nic = stats[compartment]["n_nic"]

    if np.isnan(p):
        p_text = "p = NA"
    elif p < 0.001:
        p_text = f"p = {p:.2e}"
    else:
        p_text = f"p = {p:.3f}"

    ax.set_title(
        f"{compartment}\n"
        f"MS: {n_ms} clones | NIC: {n_nic} clones\n"
        f"{p_text}"
    )


# ----------------------------
# Labels
# ----------------------------
g.set_axis_labels("Patient group", "BCR-related gene score (per clone)")

g.fig.suptitle(
    f"BCR gene module per clone (>12 cells)\n"
    f"{num_clones} shared hyperexpanded clones",
    fontsize=14,
    y=1.03,
)

plt.tight_layout()

plt.savefig(
    "..\\..\\05_plots\\08_AfterAntwerp\\BCR_module_CSF_v_PB.pdf",
    bbox_inches="tight",
    dpi=300,
)

plt.show()

In [ ]:
expr_col = "BCR_related_genes"

# ----------------------------
# Clone → donor-level aggregation (already in avg_df expected format)
# ----------------------------
avg_df = (
    sdata_deg["gex"].obs[
        ["donor", "Compartment", "catpat", expr_col]
    ]
    .groupby(["donor", "Compartment", "catpat"])[expr_col]
    .mean()
    .reset_index()
    .dropna()
)


# ----------------------------
# Stats per compartment
# ----------------------------
num_donors = avg_df["donor"].nunique()

compartment_colors = {"MS": "#4B8AC1", "NIC": "#C06A6A"}

stats = {}

for compartment, sub in avg_df.groupby("Compartment"):

    ms = sub.loc[sub["catpat"] == "MS", expr_col]
    nic = sub.loc[sub["catpat"] == "NIC", expr_col]

    n_ms = sub.loc[sub["catpat"] == "MS", "donor"].nunique()
    n_nic = sub.loc[sub["catpat"] == "NIC", "donor"].nunique()

    if len(ms) > 0 and len(nic) > 0:
        _, p = mannwhitneyu(ms, nic, alternative="two-sided")
    else:
        p = np.nan

    stats[compartment] = {"p": p, "n_ms": n_ms, "n_nic": n_nic}


# ----------------------------
# Plot
# ----------------------------
g = sb.catplot(
    data=avg_df,
    x="catpat",
    y=expr_col,
    col="Compartment",
    kind="box",
    linewidth=0.5,
    palette=compartment_colors,
    showfliers=False,
    height=5,
    aspect=0.8,
)

g.fig.set_size_inches(4.25, 5)


# ----------------------------
# Overlay donor trajectories
# ----------------------------
for ax, compartment in zip(g.axes.flat, g.col_names):

    sub = avg_df[avg_df["Compartment"] == compartment]

    sb.stripplot(
        data=sub,
        x="catpat",
        y=expr_col,
        color="black",
        jitter=0.15,
        linewidth=0.5,
        size=5,
        alpha=0.7,
        ax=ax,
        zorder=3,
    )

    # ----------------------------
    # Stats annotation
    # ----------------------------
    p = stats[compartment]["p"]
    n_ms = stats[compartment]["n_ms"]
    n_nic = stats[compartment]["n_nic"]

    if np.isnan(p):
        p_text = "p = NA"
    elif p < 0.001:
        p_text = f"p = {p:.2e}"
    else:
        p_text = f"p = {p:.3f}"

    ax.set_title(
        f"{compartment}\n"
        f"MS: {n_ms} | NIC: {n_nic}\n"
        f"{p_text}"
    )


# ----------------------------
# Labels
# ----------------------------
g.set_axis_labels(" ", "BCR module score / donor")

g.fig.suptitle(
    f"BCR-related gene module (> 12 cells)\n"
    f"{num_donors} donors",
    y=1.03,
)

plt.tight_layout()

plt.show()

# Pseudobulk - MS vs NIC - in CSF - among expanded cells (globally)

### Pseudobulked using Decoupler's version of Deseq2
https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html

The pseudo-bulk approach involves the following steps:

    Subsetting the cell type of interest

    Extracting their raw integer counts

    Summing their counts per gene into a single profile if they pass quality NIC

Then, DEA can be performed if there are at least two biological replicates per condition (more replicates are recommended).

Pseudobulking can easily be performed using the function decoupler.pp.pseudobulk(). In this example, the counts are just summed, though other modes such as the mean or any custom aggregation function are available. For more information, refer to the mode argument.

In [ ]:
sdata["gex"].obs["top_vs_bot_clones"].value_counts()

In [ ]:
## ---- Define the data ---- 
sdata_bulk = sdata.copy()

mu.pp.filter_obs(sdata_bulk["gex"], "clone_handle", lambda x: x.isin(overlap))
mu.pp.filter_obs(sdata_bulk["gex"], "donor", lambda x: ~x.isin(["Pt4", "Pt5"]))
mu.pp.filter_obs(sdata_bulk["gex"], "total_clone_count", lambda x: x > 12)
mu.pp.filter_obs(sdata_bulk["gex"], "Compartment", lambda x: x == "PB")

# Remove all unused categories
for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
    sdata_deg["gex"].obs[col] = sdata_deg["gex"].obs[col].cat.remove_unused_categories()

# Create a mask for genes to exclude
genes = sdata_bulk["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_bulk = sdata_bulk["gex"][:, genes_to_keep].copy()

## Bring the raw counts into .X
sdata_bulk.X = sdata_bulk.layers["counts"].copy()

In [ ]:
## ---- Use decoupler to pseudobulk cells ----
pdata = dc.pp.pseudobulk(
    adata=sdata_bulk,
    sample_col="donor",
    groups_col="catpat",
    mode="sum",
)

pdata

In [ ]:
pdata.obs

In [ ]:
dc.pp.filter_samples(pdata, min_cells=5, min_counts=1000)

In [ ]:
pdata.obs

In [ ]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=pdata,
    design_factors=["catpat"],
    refit_cooks=True,
    inference=inference,
)

# Compute LFCs
dds.deseq2()

# Extract contrast between conditions
stat_res = DeseqStats(dds, contrast=["catpat", "MS", "NIC"], inference=inference)

# Compute Wald test
stat_res.summary()

In [ ]:
# Extract results
results_df = stat_res.results_df

# filter
results_df = results_df[~results_df["pvalue"].isna()]
# results_df = results_df[results_df["baseMean"] > 15]
results_df = results_df[results_df.index != "AC004448.2"] #Exclude a donor-specific transcript, that is strongly expressed in select donors
results_df[results_df["padj"] < 0.05].sort_values("log2FoldChange", ascending=False).head(50)


In [ ]:
results_df.sort_values("log2FoldChange", ascending=False).head(10)

In [ ]:
dc.pl.volcano(results_df, x="log2FoldChange", y="pvalue", top=20, figsize =(6,6))
plt.savefig("../../05_plots/13_final_plots/49_MSvNIC_PB_hyperexpanded_cells_shared_in_PB_and_CSF.pdf", dpi=400, format="pdf")
plt.show()

## Compare CCL4L2

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CCL4L2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCL4L2_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CCL4L2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCL4L2_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CCL4L2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CCL4L2_clone_DEG3.pdf",
)

## Compare JUN

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="JUN",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/JUN_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="JUN",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/JUN_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="JUN",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/JUN_clone_DEG3.pdf",
)

## Compare PPP1R16B

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="PPP1R16B",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PPP1R16B_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="PPP1R16B",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PPP1R16B_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="PPP1R16B",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PPP1R16B_clone_DEG3.pdf",
)

## Compare PICALM

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="PICALM",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PICALM_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="PICALM",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PPICALM_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="PICALM",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PICALM_clone_DEG3.pdf",
)

## Compare "PUDP"

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="PUDP",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PUDP_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="PUDP",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PUDP_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="PUDP",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/PUDP_clone_DEG3.pdf",
)

## Compare "XIST"

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="XIST",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/XIST_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="XIST",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/XIST_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="XIST",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/XIST_clone_DEG3.pdf",
)

## Compare "SRGAP2"

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="SRGAP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SRGAP2_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="SRGAP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SRGAP2_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="SRGAP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SRGAP2_clone_DEG3.pdf",
)

## Compare AC013264.1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="AC013264.1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/AC013264.1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="AC013264.1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/AC013264.1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="AC013264.1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/AC013264.1_clone_DEG3.pdf",
)

## Compare TAF1D

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="TAF1D",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TAF1D_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="TAF1D",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TAF1D_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="TAF1D",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TAF1D_clone_DEG3.pdf",
)

## Compare RHOB

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="RHOB",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/RHOB_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="RHOB",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/RHOB_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="RHOB",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/RHOB_clone_DEG3.pdf",
)

## Compare ISG15

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="ISG15",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ISG15_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="ISG15",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ISG15_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="ISG15",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ISG15_clone_DEG3.pdf",
)

## Compare AC004448.2

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="AC004448.2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/AC04_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="AC004448.2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/AC004448.2_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="AC004448.2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/AC004448.2_clone_DEG3.pdf",
)

## Compare GTSF1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="GTSF1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GTSF1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="GTSF1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GTSF1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="AC004448.2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GTSF1_clone_DEG3.pdf",
)

## Compare GNG7

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="GNG7",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GNG7_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="GNG7",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GNG7_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="GNG7",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/GNG7_clone_DEG3.pdf",
)

## Compare POLR3K

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="POLR3K",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/POLR3K_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="POLR3K",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/POLR3K_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="POLR3K",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/POLR3K_clone_DEG3.pdf",
)

## Compare BCL2L1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="BCL2L1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/BCL2L1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="BCL2L1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/BCL2L1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="BCL2L1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/BCL2L1_clone_DEG3.pdf",
)

## Compare KDM6A

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="KDM6A",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/KDM6A_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="KDM6A",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/KDM6A_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="KDM6A",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/KDM6A_clone_DEG3.pdf",
)

## Compare TDP2

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="TDP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TDP2_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="TDP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TDP2_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="TDP2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TDP2_clone_DEG3.pdf",
)

# Pseudobulk - MS vs NIC - in PB - among expanded cells (globally)

### Pseudobulked using Decoupler's version of Deseq2
https://decoupler.readthedocs.io/en/latest/notebooks/scell/rna_psbk.html

The pseudo-bulk approach involves the following steps:

    Subsetting the cell type of interest

    Extracting their raw integer counts

    Summing their counts per gene into a single profile if they pass quality NIC

Then, DEA can be performed if there are at least two biological replicates per condition (more replicates are recommended).

Pseudobulking can easily be performed using the function decoupler.pp.pseudobulk(). In this example, the counts are just summed, though other modes such as the mean or any custom aggregation function are available. For more information, refer to the mode argument.

In [ ]:
sdata["gex"].obs["top_vs_bot_clones"].value_counts()

In [ ]:
## ---- Define the data ---- 
sdata_bulk = sdata.copy()

mu.pp.filter_obs(sdata_bulk["gex"], "clone_handle", lambda x: x.isin(overlap))
mu.pp.filter_obs(sdata_bulk["gex"], "donor", lambda x: ~x.isin(["Pt4", "Pt5"]))
mu.pp.filter_obs(sdata_bulk["gex"], "total_clone_count", lambda x: x > 12)
mu.pp.filter_obs(sdata_bulk["gex"], "Compartment", lambda x: x == "PB")

# Remove all unused categories
for col in sdata_deg["gex"].obs.select_dtypes(include="category").columns:
    sdata_deg["gex"].obs[col] = sdata_deg["gex"].obs[col].cat.remove_unused_categories()

# Create a mask for genes to exclude
genes = sdata_bulk["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC", "RPS", "RPL", "MT-"))

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_bulk = sdata_bulk["gex"][:, genes_to_keep].copy()

## Bring the raw counts into .X
sdata_bulk.X = sdata_bulk.layers["counts"].copy()

In [ ]:
## ---- Use decoupler to pseudobulk cells ----
pdata = dc.pp.pseudobulk(
    adata=sdata_bulk,
    sample_col="donor",
    groups_col="catpat",
    mode="sum",
)

pdata

In [ ]:
pdata.obs

In [ ]:
dc.pp.filter_samples(pdata, min_cells=5, min_counts=1000)

In [ ]:
pdata.obs

In [ ]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=pdata,
    design_factors=["catpat"],
    refit_cooks=True,
    inference=inference,
)

# Compute LFCs
dds.deseq2()

# Extract contrast between conditions
stat_res = DeseqStats(dds, contrast=["catpat", "MS", "NIC"], inference=inference)

# Compute Wald test
stat_res.summary()

In [ ]:
# Extract results
results_df = stat_res.results_df

# filter
results_df = results_df[~results_df["pvalue"].isna()]
results_df = results_df[results_df["baseMean"] > 10]
results_df = results_df[results_df.index != "AC004448.2"] #Exclude a donor-specific transcript, that is strongly expressed in select donors
results_df[results_df["padj"] < 0.05].sort_values("log2FoldChange", ascending=False).head(50)


In [ ]:
results_df.sort_values("pvalue", ascending=True).head(10)

In [ ]:
dc.pl.volcano(results_df, x="log2FoldChange", y="pvalue", top=20, figsize =(6,6))
plt.savefig("../../05_plots/13_final_plots/49_MSvNIC_PB_hyperexpanded_cells_shared_in_PB_and_CSF.pdf", dpi=400, format="pdf")
plt.show()

## Compare SLC35E1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="SLC35E1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SLC35E1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="SLC35E1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SLC35E1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="SLC35E1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/SLC35E1_clone_DEG3.pdf",
)

## Compare TNIP3

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="TNIP3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TNIP3_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="TNIP3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TNIP3_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="TNIP3",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/TNIP3_clone_DEG3.pdf",
)

## Compare YWHAH

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="YWHAH",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/YWHAH_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="YWHAH",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/YWHAH_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="YWHAH",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/YWHAH_clone_DEG3.pdf",
)

## Compare CD101

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CD101",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD101_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CD101",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD101_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CD101",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD101_clone_DEG3.pdf",
)

## Compare ST8SIA1

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="ST8SIA1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ST8SIA1_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="ST8SIA1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ST8SIA1_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="ST8SIA1",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/ST8SIA1_clone_DEG3.pdf",
)

## Compare CREM

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CREM",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CREM_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CREM",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CREM_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CREM",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CREM_clone_DEG3.pdf",
)

## Compare CD8B2

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="CD8B2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD8B2_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="CD8B2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD8B2_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="CD8B2",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/CD8B2_clone_DEG3.pdf",
)

## Compare NFKBIA

In [ ]:
plot_gene_csf_vs_pb(
    sdata,
    gene="NFKBIA",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/NFKBIA_comparison_CSF_v_PB.pdf",
)

In [ ]:
avg_df, stats = plot_gene_clone_deg(
    sdata,
    gene="NFKBIA",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/NFKBIA_clone_DEG2.pdf",
)

In [ ]:
avg_df, stats = plot_gene_donor_deg(
    sdata,
    gene="NFKBIA",
    overlap=overlap,
    save_path="../../05_plots/08_AfterAntwerp/NFKBIA_clone_DEG3.pdf",
)

In [ ]:
## ---- Define the data ---- 
sdata_bulk = sdata.copy()

mu.pp.filter_obs(sdata_bulk["gex"], "clone_handle", lambda x: x.isin(overlap))
mu.pp.filter_obs(sdata_bulk["gex"], "donor", lambda x: ~x.isin(["Pt4", "Pt5"]))
mu.pp.filter_obs(sdata_bulk["gex"], "total_clone_count", lambda x: x > 12)
mu.pp.filter_obs(sdata_bulk["gex"], "Compartment", lambda x: x == "PB")

# Remove all unused categories
sdata_bulk["gex"].obs["donor"] = sdata_bulk["gex"].obs["donor"].cat.remove_unused_categories()

fig = sc.pl.violin(sdata_bulk["gex"], keys=["CD8B2", "TNFRSF9", "XIST", "ST8SIA1", "CREM", "CD83", "NFKBIA"], size=2, groupby='donor', rotation=90)

## Relate these gene sets to previous gene sets

#### Import rat to human gene names

In [ ]:
from pybiomart import Server

# Connect to Ensembl BioMart
server = Server(host='http://www.ensembl.org')

# Get datasets
rat_dataset = server.marts['ENSEMBL_MART_ENSEMBL'].datasets['rnorvegicus_gene_ensembl']
#mmusculus_gene_ensembl

# Query orthologs
df = rat_dataset.query(attributes=[
    'external_gene_name', 
    'hsapiens_homolog_associated_gene_name'
])

df.columns = ['rat_gene', 'human_gene']

#Export this DF - update it every once in a while. 
df.to_csv(r"..\..\03_annotation\07_pybiomart_mouse_human_genes\rat_pybiomart_export.csv")

In [ ]:
pybiomart_genes = pd.read_csv(r"..\..\03_annotation\07_pybiomart_mouse_human_genes\rat_pybiomart_export.csv")

In [ ]:
pybiomart_genes.head(3)

#### Process kendirli - find the human gene counterpart

In [ ]:
kendirli_set = pd.read_excel(r"..\..\03_annotation\06_CRISPR_sets\Kendirli_Meniges_vs_blood_validation.xlsx")
# kendirli_set = pd.read_excel(r"../../03_annotation/06_CRISPR_sets/Kendirli_PvsB_validation.xlsx")
# kendirli_set = pd.read_excel(r"../../03_annotation/06_CRISPR_sets/Kendirli_validation_MvS.xlsx")
# kendirli_set = pd.read_excel(r"../../03_annotation/06_CRISPR_sets/Kendirli_validation_PvS.xlsx")
# kendirli_set = pd.read_excel(r"../../03_annotation/06_CRISPR_sets/Kendirli_validation_PvM.xlsx")

# rename
kendirli_set = kendirli_set.rename(columns = {"id":"rat_gene"})
# Select relevant columns from pybiomart_genes
df1 = pybiomart_genes[["rat_gene", "human_gene"]]

# Merge on 'rat_gene'
kendirli_set = kendirli_set.merge(df1, how="left", on="rat_gene")
kendirli_set.dropna(subset="human_gene")
kendirli_set = kendirli_set[["human_gene", "neg|p-value", "neg|rank", "neg|lfc"]]
kendirli_set = kendirli_set.rename(columns={"human_gene":"names"})
kendirli_set.head(10)

In [ ]:
kendirli_sub = kendirli_set[(kendirli_set["neg|p-value"] < 0.05) 
    & (kendirli_set["neg|lfc"] < -0.5)]
len(kendirli_sub.dropna()["names"])

# Plot CSF-biased genes with Kendirli data

In [ ]:
CSFbiased_genescore_list = CSFbiased_genescore["names"].tolist()

In [ ]:
kendirli_set["CSF_biased"] = np.where(kendirli_set["names"].isin(CSFbiased_genescore_list), "Keep", "Discard")

In [ ]:
# Keep only the genes of interest
plot_df = kendirli_set[kendirli_set["CSF_biased"] == "Keep"].copy()

# Sort by log fold change (optional)
cutoff = -np.log10(0.05)
plot_df["neg_logP"] = -np.log10(plot_df["neg|p-value"])
plot_df = plot_df.sort_values("neg_logP", ascending=False)

fig, ax = plt.subplots(figsize=(4, 3))

sb.barplot(
    data=plot_df,
    x="names",
    y="neg_logP",
    color="#000000",
    edgecolor="none",
    ax=ax,
)

# Significance cutoff (p = 0.05)
ax.axhline(
    cutoff,
    color="red",
    linestyle="--",
    linewidth=1,
)

ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("")
# ax.set_ylim(-2, 2)
ax.set_ylabel("neg|p-value")
ax.tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/50_CSFbiased_score_Kendirli_comparison.pdf")
plt.show()

In [ ]:
PBbiased_genescore_list = PBbiased_genescore["names"].tolist()

In [ ]:
kendirli_set["biased"] = np.where(kendirli_set["names"].isin(PBbiased_genescore_list), "Keep", "Discard")

In [ ]:
# Keep only the genes of interest
plot_df = kendirli_set[kendirli_set["biased"] == "Keep"].copy()

# Sort by log fold change (optional)
cutoff = -np.log10(0.05)
plot_df["neg_logP"] = -np.log10(plot_df["neg|p-value"])
plot_df = plot_df.sort_values("neg_logP", ascending=False)

fig, ax = plt.subplots(figsize=(4, 3))

sb.barplot(
    data=plot_df,
    x="names",
    y="neg_logP",
    color="#000000",
    edgecolor="none",
    ax=ax,
)

# Significance cutoff (p = 0.05)
ax.axhline(
    cutoff,
    color="red",
    linestyle="--",
    linewidth=1,
)

ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("")
# ax.set_ylim(-2, 2)
ax.set_ylabel("neg|p-value")
ax.tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.savefig("../../05_plots/13_final_plots/50_CSFbiased_score_Kendirli_comparison.pdf")
plt.show()

In [ ]:
cutoff

# Kendirli

In [ ]:
from scipy.stats import pearsonr

## Plot tissue draining lymph gene set versus ours per cell
df = sdata["gex"].obs[["Kendirli", "CSF permissive score", "total_clone_count"]]
df = df[df["total_clone_count"] > 11]

# Calculate Pearson correlation
r, p = pearsonr(df["Kendirli"], df["CSF permissive score"])

# Plot with regression line and confidence interval
plt.figure(figsize=(6, 6))
sb.regplot(
    data=df,
    x="Kendirli",
    y="CSF permissive score",
    ci=95,  # 95% confidence interval
    scatter_kws={"color": "firebrick", "s": 5},  # <-- dot size here
    line_kws={"color": "black", "linestyle": "--"}
)

# Annotate correlation
plt.title(f"All CD8 T cells\nPearson r = {r:.2f}, p = {p:.2e}")
plt.xlabel("Kendirli et al 2023")
plt.ylabel("CSF permissive score")
plt.grid(True)
plt.tight_layout()
plt.savefig("..\\..\\05_plots\\08_AfterAntwerp\\42_Correlation_cells_CSF_score_vs_Kendirli.pdf", bbox_inches='tight', format="pdf", dpi = 400)
plt.show()


In [ ]:
# Step 1: Extract relevant data
df = sdata["gex"].obs[["clone_handle", "CSF permissive score", "PB resident score", "Compartment", "Kendirli"]].copy()
df = df[df["Compartment"] == "PB"]
df = df[df["clone_handle"].isin(final_sets)]

# Step 2: Compute average score per clone
clone_avg = df.groupby("clone_handle")[["Kendirli"]].mean()

# Step 3: Reorder based on clone_order
clone_reordered = clone_avg.loc[clone_order].reset_index(drop=True)

# Step 4: Drop NaNs in CSF permissive score
clone_reordered = clone_reordered.dropna(subset=["Kendirli"])

# Prepare data
x_vals = np.arange(len(clone_reordered))
y_vals = clone_reordered["Kendirli"].values

# Compute Spearman correlation
rho, pval = spearmanr(x_vals, y_vals)

# Step 5: Plot using seaborn for regression with confidence interval
fig, ax = plt.subplots(figsize=(5, 4))

# Scatterplot + linear regression with CI
sb.regplot(x=x_vals, y=y_vals, ax=ax, ci=95, scatter_kws={"color": "firebrick", "s": 60}, line_kws={"color": "black", "linestyle": "--"})

# Horizontal dashed line at y = 0
ax.axhline(y=0, linestyle='--', linewidth=0.5, color='lightgrey')

# Add Spearman correlation text
ax.text(0.01, 0.95,
        f"Spearman ρ = {rho:.2f}\np = {pval:.2e}",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8))

# Formatting
ax.set_title("PB resident score vs clone rank\nwith Linear Regression + 95% CI and Spearman Correlation")
ax.set_xlabel("Clone ranks (sorted by % in CSF)")
ax.set_ylabel("Average Kendirli score")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.axvline(-0.5, color="black", linewidth=1)
ax.grid(False)

# Save the figure
fig.savefig("..\\..\\05_plots\\08_AfterAntwerp\\44_clone_Kendirli_score_regression_CI_spearman.pdf",
            bbox_inches='tight', format="pdf", dpi=400)

plt.show()

### Comparison to Buggert et al

In [ ]:
#Intravascular gene set from CMV-specific cells sorted via tetramer and clonally linked
Intravascular_genes = ["GZMB", "GZMH", "GNLY", "PRF1", "CCL4", "FGFBP2", "CX3CR1"]
TDL_genes = ["GZMK", "IL7R", "TCF7", "CD27", "CCR7", "NELL2"]

#Visualize
sc.tl.score_genes(sdata["gex"], 
                      Intravascular_genes,  
                      score_name='Intravascular gene set, Buggert et al 2020')


sc.tl.score_genes(sdata["gex"], 
                      TDL_genes,  
                      score_name='Tissue draining lymph gene set, Buggert et al 2020')

sc.pl.umap(
    sdata["gex"], 
    color=["Tissue draining lymph gene set, Buggert et al 2020", "Intravascular gene set, Buggert et al 2020"], 
    size=3, 
    add_outline=True,
    legend_fontoutline=2, 
    legend_fontweight='bold', 
    legend_fontsize="6", 
    color_map = 'RdYlBu_r',
    frameon=False, 
    vmin=0, 
    show=True
)


In [ ]:
from scipy.stats import pearsonr

## Plot tissue draining lymph gene set versus ours per cell
df = sdata["gex"].obs[["Tissue draining lymph gene set, Buggert et al 2020", "CSF permissive score", "total_clone_count"]]
df = df[df["total_clone_count"] > 0]

# Calculate Pearson correlation
r, p = pearsonr(df["Tissue draining lymph gene set, Buggert et al 2020"], df["CSF permissive score"])

# Plot with regression line and confidence interval
plt.figure(figsize=(6, 6))
sb.regplot(
    data=df,
    x="Tissue draining lymph gene set, Buggert et al 2020",
    y="CSF permissive score",
    ci=95,  # 95% confidence interval
    scatter_kws={"color": "black", "s": 1},  # <-- dot size here
    line_kws={"color": "blue"}
)

# Annotate correlation
plt.title(f"All CD8 T cells\nPearson r = {r:.2f}, p = {p:.3f}")
plt.xlabel("Tissue-draining lymph gene set (Buggert et al. 2020)")
plt.ylabel("CSF permissive score")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import pearsonr

## Plot tissue draining lymph gene set versus ours per cell
df = sdata["gex"].obs[["Tissue draining lymph gene set, Buggert et al 2020", "CSF permissive score", "total_clone_count"]]
df = df[df["total_clone_count"] > 11]

# Calculate Pearson correlation
r, p = pearsonr(df["Tissue draining lymph gene set, Buggert et al 2020"], df["CSF permissive score"])

# Plot with regression line and confidence interval
plt.figure(figsize=(6, 6))
sb.regplot(
    data=df,
    x="Tissue draining lymph gene set, Buggert et al 2020",
    y="CSF permissive score",
    ci=95,  # 95% confidence interval
    scatter_kws={"color": "black", "s": 1},  # <-- dot size here
    line_kws={"color": "blue"}
)

# Annotate correlation
plt.title(f"Hyperexpanded CD8 T cells (n > 11)\nPearson r = {r:.2f}, p = {p:.3f}")
plt.xlabel("Tissue-draining lymph gene set (Buggert et al. 2020)")
plt.ylabel("CSF permissive score")
plt.grid(True)
plt.tight_layout()
plt.show()

## MS versus NICs - pb cells

In [ ]:
# Copy and subset
sdata_deg = sdata.copy()

# Restrict to shared hyperexpanded clones in PB (exclude specific donors)
mu.pp.filter_obs(sdata_deg["gex"], "Compartment", lambda x: (x == "PB"))
mu.pp.filter_obs(sdata_deg["gex"], "donor", lambda x: (~x.isin(["Pt4", "Pt5"])))
mu.pp.filter_obs(sdata_deg["gex"], "total_clone_count", lambda x: (x > 11))

# Create a mask for genes to exclude
genes = sdata_deg["gex"].var_names
exclude_mask = genes.str.startswith(("TRAV", "TRBV", "TRGV", "TRDV", "TRAJ", "TRBJ", "TRGJ", "TRDJ", "TRAC", "TRBC", "TRGC", "TRDC"))
# exclude_mask = genes.str.startswith(("RPS", "RPL", "MT-"))  #Remove ribosomal genes BEFORE calculating ranks - this allows more to be in top 50

# Keep only genes not matching the exclusion
genes_to_keep = genes[~exclude_mask]
sdata_deg_filtered = sdata_deg["gex"][:, genes_to_keep]

# Run rank_genes_groups with filtered genes
sc.tl.rank_genes_groups(
    sdata_deg_filtered,
    groupby="catpat",
    groups=["MS"],
    reference="NIC",
    key_added="MS_pb_gset"
)

sc.tl.rank_genes_groups(
    sdata_deg_filtered,
    groupby="catpat",
    groups=["NIC"],
    reference="MS",
    key_added="NIC_pb_gset"
)

sc.pl.rank_genes_groups_dotplot(
      sdata_deg_filtered, groupby="catpat", standard_scale="var", n_genes=80, key="MS_pb_gset")
sc.pl.rank_genes_groups_dotplot(
      sdata_deg_filtered, groupby="catpat", standard_scale="var", n_genes=80, key="NIC_pb_gset")

# %%%% 
# Export all differentially expressed genes
# %%%%

Export_DEG = sc.get.rank_genes_groups_df(sdata_deg_filtered, group = None,key ="MS_pb_gset")
# Export_DEG.to_csv("W:\\mikrobiologie\\archiv\\AG_Schober\\Nikolaj\\R.files\\MS.Schober\\06_csv_outputs\\2025_6_CD8_MSvNIC_DEG_without_ribosomal_or_mitoch_genes.csv")


# %%% 
# Vulcano plot function
# %%%

# # From: https://www.sc-best-practices.org/conditions/differential_gene_expression.html
LOG_FOLD_CHANGE = 3
PVALUE = -np.log(0.01)  #Higher than p_adj 0.01. 
Add_labels = []
# Add_labels = ["GZMK", "CCR7", "CXCR4", "CXCR3", "ITGA4", "GPR183", "CD27"] 

def volcano_plot(adata, group_key, title=None): 
    result = sc.get.rank_genes_groups_df(adata, group=None, key=group_key).copy()
    result["-log_padj"] = -np.log(result["pvals_adj"].astype("float"))

    lowqval_de = result.loc[(abs(result["logfoldchanges"]) > LOG_FOLD_CHANGE) & (abs(result["-log_padj"]) > PVALUE)]
    other_de = result.loc[(abs(result["logfoldchanges"]) <= LOG_FOLD_CHANGE) | (abs(result["-log_padj"]) <= PVALUE)]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.grid(True, zorder=0)

    # Plot and capture the return value to rasterize later
    reg1 = sb.regplot(
        x=other_de["logfoldchanges"],
        y=other_de["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 15, "alpha": 0.5, "zorder": 2},
        ax=ax,
        color="gray"
    )
    reg2 = sb.regplot(
        x=lowqval_de["logfoldchanges"],
        y=lowqval_de["-log_padj"],
        fit_reg=False,
        scatter_kws={"s": 15, "zorder": 3},
        ax=ax,
        color="red"
    )

    # Rasterize only the PathCollection (scatter dots)
    for coll in ax.collections:
        coll.set_rasterized(True)

    # Add labels for selected genes
    labeled_genes = result[result["names"].isin(Add_labels)]
    for _, row in labeled_genes.iterrows():
        ax.text(row["logfoldchanges"], row["-log_padj"], row["names"], fontsize=8, ha='right', va='bottom', zorder=4)

    ax.set_xlabel("log2 FC")
    ax.set_ylabel("-log padj")

    if title is None:
        title = group_key.replace("_", " ")
    plt.title(title)
    plt.tight_layout()

    # Save the figure (vector text + raster dots)
    
    plt.savefig("..\\..\\05_plots\\08_AfterAntwerp\\15_CD8_vulcanoPlot_MSvNIC_bias.pdf", bbox_inches='tight', format="pdf", dpi=400)
    

    plt.show()

sc.settings.set_figure_params() #Back to default
volcano_plot(sdata_deg_filtered, group_key="MS_pb_gset", title="<-- MS & NIC -->") 


# Graveyard